# 의정이슈레이더 — 최소 실행 노트북

`src/main.py`가 하는 일(설정 로드 → Ollama 준비 → PostgreSQL/pgvector 검증 → API 서버)을
**필수 기능만** 남겨 기능별 셀로 나눴습니다. 프로젝트의 `.py` 파일을 import하지 않고 모든 코드를 셀 안에 담았습니다.

| 단계 | 기능 | 원본 위치 |
|---|---|---|
| 1 | 설정 로드 | `main.py` `load_env_file` |
| 2 | Ollama 확인·자동 시작 | `main.py` `ensure_ollama`, `agent/local_issue_agent.py` `probe_ollama` |
| 3 | PostgreSQL 연결·테이블 검증 | `api/postgres_backend.py` `validate` |
| 4 | 질의 임베딩 모델 | `api/postgres_backend.py` `_encoder`, `embed_query` |
| 5 | 회의록 하이브리드 검색 (dense + 전문검색 + RRF) | `api/postgres_backend.py` `search` |
| 6 | 대표 이슈 10개 · 자유검색 라우팅 | `api/services.py`, `config/*.csv/txt` |
| 7 | 외부 공공데이터 API 3종 (법제처·국회·KOSIS) · 타 의회 후속 흐름 | `src/src/external_policy_sources.py`, `regional_stats.py`, `external_followup.py` |
| 8 | 향후 예측: 다중 이슈 회귀 (논의 지속 확률 · Poisson 언급량, 원본 최종 선택 방법) | `src/src/20_expand_cluster_issue.py`, `22_build_forecast_features.py`, `31_build_multi_issue_panel.py`, `32_train_multi_issue_forecast.py` |
| 9 | 이슈 분석 (월별 언급·의회별 최초 관측·다음 달 예측·외부 후속 흐름) | `api/services.py` `_build_analysis` |
| 10 | 원문 근거·회의록 조회 | `api/postgres_backend.py` `get_evidence`, `get_document` |
| 11 | 로컬 LLM 채팅 (회의록 근거 + 이슈 분석 + 예측 + 외부 데이터) | `api/services.py` `chat`, `agent/local_issue_agent.py` |
| 12 | FastAPI 서버 + 대시보드 | `api/main.py`, `api/routers/*`, `api/frontend/*` |

**제외한 것**: Chroma DB, BM25/parquet 코퍼스, BGE 재순위, 이슈 군집화, 단일 이슈 예측(`23_forecast_issue.py`),
데이터 전처리·임베딩 생성 스크립트(`src/src/01~35`), LLM 질문 의도 분류 파이프라인, Streamlit 데모, 실험·평가 스크립트.

**사전 조건**: `docker compose up`으로 띄운 pgvector DB(데이터 복원 완료), 로컬 Ollama와 채팅 모델,
HuggingFace 캐시에 받아 둔 임베딩 모델 `dragonkue/snowflake-arctic-embed-l-v2.0-ko`,
(선택) 외부 공공데이터 인증값 `LAW_OC`·`ASSEMBLY_API_KEY`·`KOSIS_API_KEY`와 인터넷 연결.
설정은 같은 폴더의 `.env`, KOSIS 지표 목록은 `config/kosis_indicators.json`에서 읽습니다.

## 1. 설정 로드

In [1]:
from __future__ import annotations

import hashlib
import json
import os
import re
import shlex
import shutil
import subprocess
import threading
import time
import urllib.error
import urllib.parse
import urllib.request
from datetime import date, datetime
from pathlib import Path
from statistics import mean
from typing import Any
from urllib.parse import urlparse
from uuid import uuid4

BASE_DIR = Path.cwd()          # 이 노트북이 있는 폴더에서 실행한다


def load_env_file(path: Path) -> int:
    """KEY=VALUE 형식의 .env를 읽는다. 셸에서 이미 지정한 값은 덮어쓰지 않는다."""
    if not path.is_file():
        return 0
    loaded = 0
    for raw in path.read_text(encoding="utf-8-sig").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.removeprefix("export ").split("=", 1)
        key = key.strip()
        if key in os.environ:
            continue
        try:
            value = " ".join(shlex.split(value, comments=True, posix=True))
        except ValueError:
            value = value.strip().strip("\"'")
        os.environ[key] = value
        loaded += 1
    return loaded


print(f".env 항목 {load_env_file(BASE_DIR / '.env')}개 로드")

SETTINGS = {
    # PostgreSQL / 검색
    "dsn": os.environ.get("PGVECTOR_DSN", ""),
    "table": os.environ.get("COUNCIL_PG_TABLE", "council_arctic_raw_v1"),
    "ef_search": max(1, int(os.environ.get("COUNCIL_PG_EF_SEARCH", "200"))),
    "rrf_k": int(os.environ.get("COUNCIL_RRF_K", "60")),
    "dense_weight": float(os.environ.get("COUNCIL_DENSE_WEIGHT", "1.0")),
    "text_weight": float(os.environ.get("COUNCIL_TEXT_WEIGHT", os.environ.get("COUNCIL_BM25_WEIGHT", "1.0"))),
    # 질의 임베딩
    "embedding_model": os.environ.get("COUNCIL_EMBEDDING_MODEL", "dragonkue/snowflake-arctic-embed-l-v2.0-ko"),
    "embedding_revision": os.environ.get("COUNCIL_EMBEDDING_REVISION", "55ec6e9358a56d56af759bc8372e970caf8c305f"),
    "max_seq_length": int(os.environ.get("COUNCIL_MAX_SEQ_LENGTH", "2048")),
    "device": os.environ.get("COUNCIL_DEVICE", "auto"),
    "model_cache": os.environ.get("COUNCIL_MODEL_CACHE"),
    "allow_download": os.environ.get("COUNCIL_ALLOW_DOWNLOAD", "0") == "1",
    # 로컬 LLM
    "ollama_url": os.environ.get("OLLAMA_URL", "http://127.0.0.1:11434"),
    "auto_start_ollama": os.environ.get("AUTO_START_OLLAMA", "1").strip().lower() not in {"0", "false", "no", "off"},
    "chat_model": os.environ.get("COUNCIL_AGENT_MODEL", "exaone3.5:7.8b"),
    "chat_timeout": int(os.environ.get("COUNCIL_AGENT_TIMEOUT", "90")),
    # Ollama 기본 컨텍스트(4096 토큰)로는 근거 6건 + 분석·예측·외부 요약이 넘칠 수 있어 원본 Agent와 같은 8192로
    "chat_num_ctx": int(os.environ.get("COUNCIL_AGENT_NUM_CTX", "8192")),
    # 외부 공공데이터 (법제처·국회·KOSIS)
    "followup_mode": os.environ.get("COUNCIL_FOLLOWUP_MODE", "backtest"),        # live | backtest
    "minutes_cutoff": os.environ.get("COUNCIL_MINUTES_CUTOFF", "2025-12-31"),     # 회의록 기준일
    "external_cache": BASE_DIR / os.environ.get("COUNCIL_EXTERNAL_CACHE", "data/external/cache"),
    # API 서버
    "host": os.environ.get("HOST", "127.0.0.1"),
    "port": int(os.environ.get("PORT", "8000")),
}
if not SETTINGS["dsn"]:
    raise RuntimeError("PGVECTOR_DSN이 설정되지 않았습니다 (.env 확인)")
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", SETTINGS["table"]):
    raise ValueError("COUNCIL_PG_TABLE은 영문자·숫자·밑줄만 사용할 수 있습니다")
if SETTINGS["followup_mode"] not in {"live", "backtest"}:
    raise ValueError("COUNCIL_FOLLOWUP_MODE는 live 또는 backtest여야 합니다")
{key: value for key, value in SETTINGS.items() if key != "dsn"}

.env 항목 15개 로드


{'table': 'council_arctic_raw_v1',
 'ef_search': 200,
 'rrf_k': 60,
 'dense_weight': 1.0,
 'text_weight': 1.0,
 'embedding_model': 'dragonkue/snowflake-arctic-embed-l-v2.0-ko',
 'embedding_revision': '55ec6e9358a56d56af759bc8372e970caf8c305f',
 'max_seq_length': 2048,
 'device': 'cpu',
 'model_cache': 'C:/Users/Admin/.cache/huggingface/hub',
 'allow_download': False,
 'ollama_url': 'http://127.0.0.1:11434',
 'auto_start_ollama': True,
 'chat_model': 'hf.co/mykor/A.X-4.0-Light-gguf:Q4_K_M',
 'chat_timeout': 240,
 'chat_num_ctx': 8192,
 'followup_mode': 'live',
 'minutes_cutoff': '2025-12-31',
 'external_cache': WindowsPath('C:/TEST/team-ggeodeureok_exaone/notebook_minimal/data/external/cache'),
 'host': '127.0.0.1',
 'port': 8010}

## 2. Ollama 확인·자동 시작
회의록이 외부로 나가지 않도록 localhost Ollama만 허용합니다. 서버가 꺼져 있으면 `ollama serve`를 백그라운드로 띄웁니다.

In [2]:
LOCAL_HOSTS = {"localhost", "127.0.0.1", "::1"}


def probe_ollama(url: str, model: str, timeout: float = 3.0) -> dict[str, Any]:
    """Ollama 서버 응답 여부와 모델 설치 여부를 확인한다."""
    if (urlparse(url).hostname or "").lower() not in LOCAL_HOSTS:
        return {"reachable": False, "model_available": False, "models": [],
                "error": "보안 정책상 localhost Ollama만 사용할 수 있습니다."}
    try:
        with urllib.request.urlopen(url.rstrip("/") + "/api/tags", timeout=timeout) as response:
            payload = json.loads(response.read().decode("utf-8"))
    except Exception as error:
        return {"reachable": False, "model_available": False, "models": [],
                "error": f"{type(error).__name__}: {error}"}
    names = sorted({str(v) for item in payload.get("models") or []
                    for v in (item.get("name"), item.get("model")) if v})
    available = model in names or (":" not in model and any(n.split(":", 1)[0] == model for n in names))
    return {"reachable": True, "model_available": available, "models": names,
            "error": None if available else f"로컬 모델 '{model}'이 설치되지 않았습니다."}


def _tail(path: Path, lines: int = 40) -> str:
    try:
        return "\n".join(path.read_text(encoding="utf-8", errors="replace").splitlines()[-lines:])
    except OSError:
        return "(로그 없음)"


def ensure_ollama(url: str, wait_seconds: float = 15.0) -> bool:
    if probe_ollama(url, "")["reachable"]:
        print("Ollama 서버가 이미 실행 중입니다.")
        return True
    if not SETTINGS["auto_start_ollama"]:
        print("AUTO_START_OLLAMA 설정으로 자동 시작이 꺼져 있습니다.")
        return False
    parsed = urlparse(url)
    if parsed.scheme != "http" or (parsed.hostname or "").lower() not in LOCAL_HOSTS:
        raise RuntimeError(f"로컬 Ollama 자동 시작에 사용할 수 없는 주소입니다: {url}")
    executable = shutil.which("ollama")
    if not executable:
        raise RuntimeError("ollama 실행 파일을 찾을 수 없습니다")
    host = f"[{parsed.hostname}]" if ":" in parsed.hostname else parsed.hostname
    env = dict(os.environ, OLLAMA_HOST=f"{host}:{parsed.port or 11434}")
    log_path = BASE_DIR / "logs" / "ollama.log"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as log_file:
        options: dict[str, Any] = {"env": env, "stdout": log_file, "stderr": subprocess.STDOUT}
        if os.name == "nt":
            options["creationflags"] = getattr(subprocess, "CREATE_NO_WINDOW", 0)
        else:
            options["start_new_session"] = True
        subprocess.Popen([executable, "serve"], **options)
    print(f"Ollama 서버 시작 요청 (로그: {log_path})")
    deadline = time.monotonic() + wait_seconds
    while time.monotonic() < deadline:
        if probe_ollama(url, "", timeout=0.75)["reachable"]:
            return True
        time.sleep(0.5)
    print(_tail(log_path))
    return False


if not ensure_ollama(SETTINGS["ollama_url"]) and SETTINGS["auto_start_ollama"]:
    raise RuntimeError("Ollama가 제한 시간 안에 응답하지 않았습니다")
status = probe_ollama(SETTINGS["ollama_url"], SETTINGS["chat_model"])
print("Ollama 연결:", status["reachable"], "| 채팅 모델", SETTINGS["chat_model"], "설치:", status["model_available"])
if not status["model_available"]:
    print(f"  → 터미널에서 `ollama pull {SETTINGS['chat_model']}` 실행")

Ollama 서버가 이미 실행 중입니다.


Ollama 연결: True | 채팅 모델 hf.co/mykor/A.X-4.0-Light-gguf:Q4_K_M 설치: True


## 3. PostgreSQL 연결·테이블 검증

In [3]:
import psycopg
from pgvector.psycopg import register_vector
from psycopg.rows import dict_row

TABLE = SETTINGS["table"]
FIELDS = ("chunk_id, parent_chunk_id, document_id, council, meeting_date, year, "
          "committee, meeting_name, agenda_title, priority, document, metadata")


def connect():
    connection = psycopg.connect(SETTINGS["dsn"], row_factory=dict_row)
    register_vector(connection)
    return connection


with connect() as conn:
    conn.execute(f"SELECT {FIELDS}, embedding FROM {TABLE} LIMIT 0")   # 필요한 열이 모두 있는지
    row = conn.execute(
        f"SELECT vector_dims(embedding) AS dims FROM {TABLE} WHERE embedding IS NOT NULL LIMIT 1").fetchone()
    approx_rows = conn.execute(
        "SELECT reltuples::bigint AS n FROM pg_class WHERE relname = %s", (TABLE,)).fetchone()["n"]
if row is None:
    raise RuntimeError(f"pgvector 테이블 {TABLE}에 임베딩 행이 없습니다")
VECTOR_DIMS = int(row["dims"])
print(f"테이블 {TABLE}: 약 {approx_rows:,}행, 벡터 {VECTOR_DIMS}차원")

테이블 council_arctic_raw_v1: 약 900,810행, 벡터 1024차원


## 4. 질의 임베딩 모델
DB에 저장된 청크 임베딩과 같은 모델(arctic-embed-ko, 고정 revision)로 질문을 벡터화합니다.
`COUNCIL_ALLOW_DOWNLOAD=1`이 아니면 로컬 캐시에서만 읽습니다.

In [4]:
import torch
from sentence_transformers import SentenceTransformer

device = SETTINGS["device"]
if device == "auto":
    device = "cuda" if torch.cuda.is_available() else "cpu"
encoder_kwargs: dict[str, Any] = {
    "device": device,
    "local_files_only": not SETTINGS["allow_download"],
    "revision": SETTINGS["embedding_revision"],
}
if SETTINGS["model_cache"]:
    encoder_kwargs["cache_folder"] = str(Path(SETTINGS["model_cache"]).expanduser().resolve())
ENCODER = SentenceTransformer(SETTINGS["embedding_model"], **encoder_kwargs)
ENCODER.max_seq_length = SETTINGS["max_seq_length"]


def embed_query(text: str) -> list[float]:
    kwargs: dict[str, Any] = {"convert_to_numpy": True, "normalize_embeddings": True, "show_progress_bar": False}
    if "query" in getattr(ENCODER, "prompts", {}):
        kwargs["prompt_name"] = "query"
    return ENCODER.encode([text], **kwargs)[0].astype("float32").tolist()


dims = len(embed_query("의정 이슈 검색 시작 점검"))
if dims != VECTOR_DIMS:
    raise RuntimeError(f"질의 임베딩과 DB 벡터 차원이 다릅니다: {dims} != {VECTOR_DIMS}")
print(f"임베딩 모델 로드 완료 ({device}, {dims}차원)")

C:\Users\Admin\miniconda3\envs\py311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


Loading weights:   5%|▌         | 21/391 [00:00<00:01, 202.34it/s]


Loading weights:  11%|█         | 42/391 [00:00<00:01, 182.33it/s]


Loading weights:  16%|█▌        | 61/391 [00:00<00:02, 153.33it/s]


Loading weights:  20%|█▉        | 77/391 [00:00<00:02, 152.76it/s]


Loading weights:  24%|██▍       | 93/391 [00:00<00:02, 137.75it/s]


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 602.85it/s]

임베딩 모델 로드 완료 (cpu, 1024차원)


## 5. 회의록 하이브리드 검색
- **dense**: pgvector HNSW 코사인 거리 상위 후보
- **전문검색**: `websearch_to_tsquery('simple', …)`로 질문 단어 중 하나라도 들어간 청크 (단어를 OR로 묶음)
- **RRF**: 두 순위를 점수 척도와 무관하게 `1/(60+순위)`로 합산

> ⚠️ 이 테이블에는 전문검색용 GIN 인덱스가 없어 전문검색 쿼리가 전체 행을 훑습니다(쿼리당 수 분).
> 아래 셀의 `CREATE_FTS_INDEX = True`로 한 번 실행하면 인덱스를 만들어 빨라집니다. **DB를 변경**하므로 기본값은 꺼 둡니다.

In [5]:
CREATE_FTS_INDEX = False   # True: 전문검색 GIN 인덱스 생성 (DB 변경, 최초 1회 수 분~수십 분)

if CREATE_FTS_INDEX:
    with psycopg.connect(SETTINGS["dsn"], autocommit=True) as conn:
        conn.execute(f"CREATE INDEX IF NOT EXISTS {TABLE}_fts_idx ON {TABLE} "
                     "USING gin (to_tsvector('simple', coalesce(document, '')))")
    print("전문검색 인덱스 준비 완료")

In [6]:
def full_text_query(text: str) -> str:
    """질문 단어를 OR로 묶어 한 단어가 없어도 결과가 사라지지 않게 한다."""
    terms = list(dict.fromkeys(re.findall(r"[0-9A-Za-z가-힣]+", text)))
    return " OR ".join(f'"{term}"' for term in terms)


def where_clause(filters: dict[str, Any] | None) -> tuple[str, list[Any]]:
    clauses, values = [], []
    for key, column, op in (("council", "council", "="), ("year", "year", "="),
                            ("committee", "committee", "ILIKE"),
                            ("date_from", "meeting_date", ">="), ("date_to", "meeting_date", "<=")):
        value = (filters or {}).get(key)
        if value is None:
            continue
        clauses.append(f"{column} {op} %s")
        values.append(f"%{value}%" if op == "ILIKE" else value)
    return (" AND " + " AND ".join(clauses)) if clauses else "", values


def rrf_fuse(dense_rows: list[dict], text_rows: list[dict], limit: int) -> list[dict]:
    """가중 RRF: 점수 = Σ weight / (rrf_k + 순위)"""
    fused: dict[str, dict] = {}
    k = SETTINGS["rrf_k"]
    for source, rows, weight in (("dense", dense_rows, SETTINGS["dense_weight"]),
                                 ("text", text_rows, SETTINGS["text_weight"])):
        for rank, row in enumerate(rows, start=1):
            if not row.get("chunk_id"):
                continue
            item = fused.setdefault(row["chunk_id"], {**row, "retrieval_score": 0.0, "retrieval_sources": []})
            item["retrieval_score"] += weight / (k + rank)
            item["retrieval_sources"].append(source)
            if source == "dense":
                item["similarity"] = row.get("similarity")
    ordered = sorted(fused.values(), key=lambda r: (-r["retrieval_score"], str(r["chunk_id"])))[:limit]
    for rank, row in enumerate(ordered, start=1):
        row["rank"] = rank
    return ordered


def public_row(row: dict[str, Any]) -> dict[str, Any]:
    """DB 행을 API 응답 형태로 바꾼다 (메타데이터 병합, 날짜 문자열화)."""
    metadata = row.get("metadata") or {}
    if isinstance(metadata, str):
        try:
            metadata = json.loads(metadata)
        except json.JSONDecodeError:
            metadata = {}
    metadata = dict(metadata) if isinstance(metadata, dict) else {}
    for field in ("council", "meeting_date", "year", "committee", "meeting_name", "agenda_title",
                  "priority", "parent_chunk_id", "document_id"):
        value = row.get(field)
        if value is not None:
            metadata.setdefault(field, value.isoformat() if isinstance(value, (date, datetime)) else value)
    return {
        "rank": row.get("rank"), "chunk_id": str(row.get("chunk_id") or ""),
        "parent_chunk_id": row.get("parent_chunk_id"), "document_id": row.get("document_id"),
        "similarity": row.get("similarity"), "retrieval_score": row.get("retrieval_score"),
        "retrieval_sources": row.get("retrieval_sources") or [],
        "text": str(row.get("document") or ""), "metadata": metadata, "parent_context": None,
    }


def search(query: str, top_k: int = 10, candidate_k: int | None = None,
           filters: dict[str, Any] | None = None) -> dict[str, Any]:
    started = time.perf_counter()
    vector = embed_query(query)
    embedding_ms = (time.perf_counter() - started) * 1000
    candidate_k = max(top_k, candidate_k or top_k * 5)
    where_sql, where_values = where_clause(filters)
    with connect() as conn:
        conn.execute("SELECT set_config('hnsw.ef_search', %s, true)",
                     (str(max(SETTINGS["ef_search"], candidate_k)),))
        dense_rows = conn.execute(f"""
            SELECT {FIELDS}, 1 - (embedding <=> %s::vector) AS similarity
            FROM {TABLE} WHERE embedding IS NOT NULL{where_sql}
            ORDER BY embedding <=> %s::vector, chunk_id LIMIT %s""",
            [vector, *where_values, vector, candidate_k]).fetchall()
        # 단어가 적으면 점수 동점이 많아 chunk_id로 순서를 고정한다 (검색창·채팅 결과 일치)
        text_rows = conn.execute(f"""
            WITH q AS (SELECT websearch_to_tsquery('simple', %s) AS term)
            SELECT {FIELDS}, ts_rank_cd(to_tsvector('simple', coalesce(document, '')), q.term) AS text_rank_score
            FROM {TABLE}, q
            WHERE to_tsvector('simple', coalesce(document, '')) @@ q.term{where_sql}
            ORDER BY text_rank_score DESC, chunk_id LIMIT %s""",
            [full_text_query(query), *where_values, candidate_k]).fetchall()
    fused = rrf_fuse([dict(r) for r in dense_rows], [dict(r) for r in text_rows], top_k)
    return {
        "query": query, "backend": "postgresql_pgvector_full_text", "retrieval_mode": "hybrid_rrf",
        "result_count": len(fused), "results": [public_row(r) for r in fused],
        "search_ms": round((time.perf_counter() - started) * 1000, 3),
        "query_embedding_ms": round(embedding_ms, 3),
    }

검색 실행 예시:

In [7]:
result = search("전세사기 피해자 지원 대책", top_k=5)
print(f"{result['result_count']}건, {result['search_ms'] / 1000:.1f}초")
for row in result["results"]:
    m = row["metadata"]
    print(f"{row['rank']}. [{'+'.join(row['retrieval_sources'])}] {m.get('council')} {m.get('meeting_date')} "
          f"{m.get('meeting_name')} | {row['text'][:60]}")

5건, 25.8초
1. [text] 서울특별시의회 2025-11-10 보건복지위원회 회의록 제333회 | [미상 김인제위원] 자치구별 신청 베이스에 귀속된 결과라고 하기에는 우리가 2024년도에 전세사기 피해자들을
2. [dense] 전라남도의회 2025-03-11 본회의 회의록 제388회(1차) | [의원 박경미] 전세사기 피해는 아직도 계속되고 있지만 피해자들의 실질적인 대책은 미비한 실정입니다.서울, 
3. [text] 서울특별시의회 2025-11-10 보건복지위원회 회의록 제333회 | [미상 김인제위원] 자치구별 신청 베이스에 귀속된 결과라고 하기에는 우리가 2024년도에 전세사기 피해자들을
4. [dense] 전라남도의회 2025-03-11 본회의 회의록 제388회(1차) | [의원 박경미] 전세사기 피해는 아직도 계속되고 있지만 피해자들의 실질적인 대책은 미비한 실정입니다.서울, 
5. [text] 서울특별시의회 2025-11-10 보건복지위원회 회의록 제333회 | [미상 김인제위원] 우리 전세사기 피해자 자치구별 현황을 보면 서울시가 전체 자치구별 신청 현황을 조사한 겁


## 6. 대표 이슈 10개 · 자유검색 라우팅
원본의 `config/representative_issue_keys_v1.txt` + `issue_seed_catalog_v1.csv`에서 사용하던 10개 이슈입니다.
자유검색어가 이슈명·별칭과 충분히 겹치면(점수 ≥ 0.5) 해당 고정 이슈로 연결합니다.

In [8]:
ISSUES = [
    # (issue_key, issue_name, 검색 query, 별칭, 분야) — 원본 issue_seed_catalog_v1.csv의 행 순서
    ("birth_rate", "저출생·출산율 대응 정책", "출산율 저하 대응 정책", "저출산 대응|저출생 정책|출산 지원", "보건/복지"),
    ("population_decline", "인구감소·지역소멸 대응", "인구감소 지역소멸 대응 정책", "지방소멸|인구 유출|생활인구", "자치/행정"),
    ("youth_jobs", "청년 일자리와 지역정착 지원", "청년 일자리 지역정착 지원 정책", "청년고용|청년정책|청년 유출", "경제/산업"),
    ("jeonse_fraud", "전세사기 피해 예방과 지원", "전세사기 피해자 지원 대책", "전세 피해|임차인 보호|주거 피해", "도시/교통"),
    ("regional_healthcare", "지역의료 공백과 필수의료 대응", "지역의료 공백 필수의료 대책", "의료인력|공공의료|응급의료", "보건/복지"),
    ("care_policy", "돌봄 공백과 통합돌봄 정책", "돌봄 공백 통합돌봄 지원 정책", "복지 돌봄|가족 돌봄|사회서비스", "보건/복지"),
    ("school_violence", "학교폭력 예방과 피해자 보호", "학교폭력 예방 피해자 보호 대책", "학폭|학생 안전|피해 학생", "교육/문화"),
    ("climate_neutrality", "기후위기와 탄소중립 전환", "기후위기 탄소중립 대응 정책", "온실가스|기후변화|탄소 감축", "환경/안전"),
    ("flood_safety", "집중호우·침수 재난 대응", "집중호우 침수 피해 예방 대책", "풍수해|하천 범람|침수 예방", "환경/안전"),
    ("small_business", "소상공인 경영 안정 지원", "소상공인 경영 안정 지원 정책", "자영업자|골목상권|경영 부담", "경제/산업"),
]
ISSUE_LIST = [
    {"issue_key": key, "issue_name": name, "query": query, "aliases": aliases.split("|"), "category": category}
    for key, name, query, aliases, category in ISSUES
]
ISSUE_BY_KEY = {issue["issue_key"]: issue for issue in ISSUE_LIST}


def _terms(text: str) -> set[str]:
    return {t for t in re.sub(r"[^0-9A-Za-z가-힣]+", " ", text.casefold()).split() if len(t) >= 2}


def route_fixed_issue(question: str) -> tuple[dict | None, float]:
    """질문과 가장 가까운 대표 이슈를 보수적인 문자열 점수로 고른다."""
    text, question_terms = question.casefold(), _terms(question)
    best, best_score = None, 0.0
    for issue in ISSUE_LIST:
        score = 0.0
        for name in [issue["issue_name"], *issue["aliases"]]:
            name = name.strip().casefold()
            if name and name in text:
                score = max(score, 1.0)
                continue
            terms = _terms(name)
            if terms:
                score = max(score, len(question_terms & terms) / len(terms))
                if any(len(t) >= 3 and t in question_terms for t in terms):
                    score = max(score, 0.75)
        if score > best_score:
            best, best_score = issue, score
    return (best, best_score) if best_score >= 0.5 else (None, best_score)


for q in ["전세사기 대책", "학폭 피해 학생", "드론 배송 규제"]:
    issue, score = route_fixed_issue(q)
    print(f"{q!r:16} → {issue['issue_key'] if issue else '동적 분석'} (점수 {score:.2f})")

'전세사기 대책'        → jeonse_fraud (점수 0.75)
'학폭 피해 학생'       → school_violence (점수 1.00)
'드론 배송 규제'       → 동적 분석 (점수 0.00)


## 7. 외부 공공데이터 API 3종 (법제처 · 열린국회 · KOSIS)
회의록 기반 분석을 **참고용(reference_only)** 으로 보강합니다. 회의록 원문·질문 원문은 외부로 보내지 않습니다.

- 외부로 나가는 값: `check_keyword`를 통과한 **짧은 정책어**(1~30자, 4단어 이하, 사람 이름 차단)와 KOSIS 통계표 ID뿐
- 허용 호스트: `www.law.go.kr`, `open.assembly.go.kr`, `kosis.kr` / 모든 호출은 `access_log.jsonl`에 기록(인증값 제외)
- 응답은 로컬 캐시(`COUNCIL_EXTERNAL_CACHE`)에 저장해 재사용 (조례·의안 7일, KOSIS 30일)
- 인증값: `.env`의 `LAW_OC`(법제처), `ASSEMBLY_API_KEY`(국회), `KOSIS_API_KEY`(통계청). 없는 출처는 건너뜁니다.

### 7-1. 공통: 검색어 검사 · 캐시 · 접근 기록 · 지역 매핑

In [9]:
import html.entities
import statistics
import xml.etree.ElementTree as ET
from collections import defaultdict

DATA_SOURCES = {
    "law_ordin": {
        "name": "국가법령정보 공동활용 Open API — 자치법규 목록 조회",
        "provider": "법제처 (국가법령정보센터)",
        "api": "https://www.law.go.kr/DRF/lawSearch.do?target=ordin (DRF 목록 조회 서비스, type=XML)",
        "version": "DRF 목록 조회 서비스 (법제처가 버전 번호를 따로 부여하지 않아 서비스명·조회일로 표기)",
        "data_type": "공공데이터 — 자치법규 이름·지자체기관명·공포일자·제개정 구분 (본문 미수집)",
        "copyright": "조례·규칙은 저작권법 제7조(보호받지 못하는 저작물)에 해당. Open API 이용은 법제처 공동활용 이용약관을 따름",
        "terms": "https://open.law.go.kr",
        "attribution": "출처: 법제처 국가법령정보센터(www.law.go.kr) 자치법규 Open API",
        "key_env": "LAW_OC",
    },
    "assembly_bills": {
        "name": "열린국회정보 Open API — 국회의원 발의법률안 (서비스 ID nzmimeepazxkubdpn)",
        "provider": "국회사무처 (열린국회정보)",
        "api": "https://open.assembly.go.kr/portal/openapi/nzmimeepazxkubdpn",
        "version": "Open API 서비스 nzmimeepazxkubdpn (제21·22대 국회 조회)",
        "data_type": "공공데이터 — 의안명·대표발의자·제안일·처리결과",
        "copyright": "열린국회정보 이용약관 및 해당 데이터의 공공누리 표시를 따름 (이용 전 확인 필요)",
        "terms": "https://open.assembly.go.kr",
        "attribution": "출처: 국회사무처 열린국회정보(open.assembly.go.kr)",
        "key_env": "ASSEMBLY_API_KEY",
    },
    "kosis": {
        "name": "KOSIS 국가통계포털 공유서비스 Open API — 통계자료(statisticsParameterData)·통계표 검색(statisticsSearch)",
        "provider": "통계청 (각 통계표의 작성기관은 결과에 따로 표시)",
        "api": "https://kosis.kr/openapi/Param/statisticsParameterData.do , https://kosis.kr/openapi/statisticsSearch.do",
        "version": "KOSIS 공유서비스 Open API — 통계표별 버전은 '통계표ID + 수록시점 + 조회일'로 결과마다 표시",
        "data_type": "공공데이터 — 17개 시도별 집계 통계값(개인 단위 자료 아님)",
        "copyright": "KOSIS 이용약관 및 각 통계표의 작성기관 이용 조건(공공누리 표시 등)을 따름 — 이용 전 확인 필요",
        "terms": "https://kosis.kr/openapi",
        "attribution": "출처: 통계청 KOSIS 국가통계포털(kosis.kr) — 통계표명·작성기관·수록시점은 결과별로 표시",
        "key_env": "KOSIS_API_KEY",
    },
}
ALLOWED_HOSTS = {"www.law.go.kr", "open.assembly.go.kr", "kosis.kr"}
USER_AGENT = {"User-Agent": "local-council-minutes-research/1.0"}
CACHE_DIR = Path(SETTINGS["external_cache"])

# 광역의회 ↔ 자치법규 지자체기관명(정식 명칭)
REGION = {
    "서울특별시의회": ["서울특별시"], "부산광역시의회": ["부산광역시"], "대구광역시의회": ["대구광역시"],
    "인천광역시의회": ["인천광역시"], "광주광역시의회": ["광주광역시"], "대전광역시의회": ["대전광역시"],
    "울산광역시의회": ["울산광역시"], "세종특별자치시의회": ["세종특별자치시"], "경기도의회": ["경기도"],
    "강원특별자치도의회": ["강원특별자치도", "강원도"], "충청북도의회": ["충청북도"], "충청남도의회": ["충청남도"],
    "전북특별자치도의회": ["전북특별자치도", "전라북도"], "전라남도의회": ["전라남도"], "경상북도의회": ["경상북도"],
    "경상남도의회": ["경상남도"], "제주특별자치도의회": ["제주특별자치도"],
}
COUNCIL_ALIASES = {"전라북도의회": "전북특별자치도의회", "강원도의회": "강원특별자치도의회"}   # 회의록의 옛 명칭


class PolicyError(ValueError):
    """외부 데이터 이용 원칙에 어긋나는 요청"""


_KEYWORD_OK = re.compile(r"^[가-힣A-Za-z0-9][가-힣A-Za-z0-9 ·\-]{0,29}$")
_PERSON = re.compile(r"[가-힣]{2,4}\s*(의원|위원장|위원|의장|국장|실장|과장|시장|도지사|지사|교육감|교육장|단장|본부장|청장)(님)?$"
                     r"|^(위원장|부위원장|부의장|의원|위원|의장|국장|실장|과장|시장|도지사|교육감)\s*[가-힣]{2,4}$")


def check_keyword(keyword: str) -> str:
    """외부로 보낼 검색어 검사: 짧은 정책어만 허용 (회의록 문장·사람 이름 유출 방지)"""
    k = re.sub(r"\s+", " ", (keyword or "").strip())
    if not _KEYWORD_OK.match(k) or len(k.split()) > 4:
        raise PolicyError(f"외부로 보낼 수 없는 검색어입니다(짧은 정책어 1~30자, 4단어 이하만 허용): {k[:40]!r}")
    if _PERSON.search(k):
        raise PolicyError(f"사람(발언자) 이름으로 보이는 검색어는 외부로 보내지 않습니다: {k!r}")
    return k


def api_key(env: str) -> str | None:
    return (os.environ.get(env) or "").strip() or None


def key_available(source_id: str) -> bool:
    return api_key(DATA_SOURCES[source_id]["key_env"]) is not None


def sources_used(keys) -> list[dict]:
    fields = ("name", "provider", "version", "copyright", "terms", "attribution")
    return [{"id": k, **{f: DATA_SOURCES[k][f] for f in fields}} for k in keys if k in DATA_SOURCES]


def _cache_file(key: str) -> Path:
    return CACHE_DIR / (hashlib.md5(key.encode("utf-8")).hexdigest() + ".json")


def cache_get(key: str, max_age_days: int = 7):
    path = _cache_file(key)
    if path.exists() and (time.time() - path.stat().st_mtime) < max_age_days * 86400:
        return json.loads(path.read_text(encoding="utf-8"))
    return None


def cache_put(key: str, value) -> None:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    _cache_file(key).write_text(json.dumps(
        {"key": key, "saved_at": datetime.now().isoformat(timespec="seconds"), "value": value},
        ensure_ascii=False), encoding="utf-8")


def external_get(url: str, source: str, keyword: str, timeout: int = 20) -> bytes:
    """허용 호스트만 호출하고, 호출 사실(인증값 제외)을 access_log.jsonl에 남긴다."""
    host = urlparse(url).hostname
    if host not in ALLOWED_HOSTS:
        raise PolicyError(f"허용되지 않은 외부 주소: {host}")
    with urllib.request.urlopen(urllib.request.Request(url, headers=USER_AGENT), timeout=timeout) as response:
        body = response.read()
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    with (CACHE_DIR / "access_log.jsonl").open("a", encoding="utf-8") as log:
        log.write(json.dumps({"at": datetime.now().isoformat(timespec="seconds"), "source": source,
                              "host": host, "keyword": keyword, "bytes": len(body)}, ensure_ascii=False) + "\n")
    return body


def safe_call(func, *args, **kwargs) -> dict:
    """정책 위반·외부 서버 오류가 나도 분석을 멈추지 않고 error 결과로 돌려준다."""
    try:
        return func(*args, **kwargs)
    except PolicyError as error:
        return {"error": f"이용 원칙 위반으로 차단: {error}"}
    except Exception as error:
        return {"error": f"{type(error).__name__}: {error}"}


def normalize_council(council: str) -> str:
    name = (council or "").strip()
    return COUNCIL_ALIASES.get(name, name)


def region_of_org(org: str) -> tuple[str | None, str | None]:
    """자치법규 지자체기관명 → (광역의회, 단계) — 단계: '광역'(본청·교육청) / '기초'(시·군·구)"""
    name = (org or "").strip()
    for council, forms in REGION.items():
        for form in forms:
            if name.startswith(form):
                rest = name[len(form):].strip()
                return council, ("광역" if rest in ("", "교육청") else "기초")
    return None, None


print({source: key_available(source) for source in DATA_SOURCES})

{'law_ordin': True, 'assembly_bills': True, 'kosis': True}


### 7-2. 법제처 자치법규 (조례·규칙 이름 검색)

In [10]:
_XML_STANDARD = {b"amp", b"lt", b"gt", b"apos", b"quot"}
_XML_NAMED = re.compile(rb"&([A-Za-z][A-Za-z0-9]+);")
_XML_BARE_AMP = re.compile(rb"&(?!#\d+;|#x[0-9A-Fa-f]+;|amp;|lt;|gt;|apos;|quot;)")


def _xml_safe(payload: bytes) -> bytes:
    """법제처 XML에 섞인 HTML 엔티티(&nbsp; 등)와 맨 &를 XML이 읽을 수 있게 바꾼다."""
    def replace(match: re.Match[bytes]) -> bytes:
        name = match.group(1)
        if name in _XML_STANDARD:
            return match.group(0)
        value = html.entities.html5.get(name.decode("ascii") + ";")
        if value is None:
            return b"&amp;" + name + b";"
        return "".join(f"&#{ord(ch)};" for ch in value).encode("ascii")
    return _XML_BARE_AMP.sub(b"&amp;", _XML_NAMED.sub(replace, payload))


def fetch_local_ordinances(keyword: str, max_pages: int = 5) -> dict:
    """자치법규 '이름'에 정책어가 들어간 목록 (본문은 받지 않음)"""
    src = DATA_SOURCES["law_ordin"]
    oc = api_key(src["key_env"])
    if not oc:
        return {"source": src["name"], "error": f"{src['key_env']}가 설정되지 않아 사용할 수 없음 ({src['terms']})"}
    keyword = check_keyword(keyword)
    cache_key = f"ordin|{keyword}|{hashlib.sha256(oc.encode()).hexdigest()[:8]}"
    items = (cache_get(cache_key) or {}).get("value")
    if items is None:
        items = []
        for page in range(1, max_pages + 1):
            query = urllib.parse.urlencode({"OC": oc, "target": "ordin", "type": "XML", "query": keyword,
                                            "display": 100, "page": page})
            root = ET.fromstring(_xml_safe(external_get(
                f"https://www.law.go.kr/DRF/lawSearch.do?{query}", "law_ordin", keyword)))
            if str(root.tag).lower().endswith("html"):
                raise RuntimeError("법제처 API가 XML 대신 HTML 오류 페이지를 반환했습니다")
            if (root.findtext("resultCode") or "00") != "00":
                return {"source": src["name"], "error": f"법제처 API 오류: {root.findtext('resultMsg')}"}
            rows = root.findall("law")
            for row in rows:
                get = lambda tag: (row.findtext(tag) or "").strip()
                day = get("공포일자")
                items.append({"name": get("자치법규명"), "org": get("지자체기관명"), "kind": get("자치법규종류"),
                              "type": get("제개정구분명"),
                              "promulgated": f"{day[:4]}-{day[4:6]}-{day[6:8]}" if len(day) == 8 else None,
                              "law_id": get("자치법규ID")})
            if page * 100 >= int(root.findtext("totalCnt") or 0) or not rows:
                break
            time.sleep(0.5)
        cache_put(cache_key, items)
    out = [{**item, **dict(zip(("council", "level"), region_of_org(item["org"])))} for item in items]
    return {"source": src["name"], "source_id": "law_ordin", "attribution": src["attribution"],
            "keyword": keyword, "count": len(out), "items": out}


demo = safe_call(fetch_local_ordinances, "전세사기")
print(demo.get("error") or f"'{demo['keyword']}' 자치법규 {demo['count']}건")
for item in (demo.get("items") or [])[:3]:
    print("  ", item["promulgated"], item["org"], "|", item["name"], f"({item['council']} {item['level']})")

'전세사기' 자치법규 88건
   2023-11-06 전남광주통합특별시 강진군 | 강진군 전세사기피해자 등 주택임차인 보호 및 지원 조례 (None None)
   2025-08-12 경기도 | 경기도 전세사기 예방 및 안전전세 관리단 운영 조례 (경기도의회 광역)
   2025-10-10 경기도 | 경기도 주택임차인 전세피해 지원 조례 (경기도의회 광역)


### 7-3. 열린국회정보 국회의원 발의 법률안 (제21·22대)

In [11]:
def fetch_national_bills(keyword: str) -> dict:
    src = DATA_SOURCES["assembly_bills"]
    key = api_key(src["key_env"])
    if not key:
        return {"source": src["name"], "error": f"{src['key_env']}가 설정되지 않아 사용할 수 없음 ({src['terms']})"}
    keyword = check_keyword(keyword)
    out = []
    for age in (22, 21):
        cache_key = f"bills|{age}|{keyword}"
        data = (cache_get(cache_key) or {}).get("value")
        if data is None:
            query = urllib.parse.urlencode({"KEY": key, "Type": "json", "pIndex": 1, "pSize": 100,
                                            "AGE": age, "BILL_NAME": keyword})
            data = json.loads(external_get(
                f"https://open.assembly.go.kr/portal/openapi/nzmimeepazxkubdpn?{query}", "assembly_bills", keyword))
            cache_put(cache_key, data)
        blocks = data.get("nzmimeepazxkubdpn") if isinstance(data, dict) else None
        for row in (blocks[-1].get("row", []) if blocks else []):
            out.append({"bill": row.get("BILL_NAME"), "proposer": row.get("PROPOSER"),
                        "date": (row.get("PROPOSE_DT") or "")[:10], "result": row.get("PROC_RESULT"), "age": age})
    return {"source": src["name"], "source_id": "assembly_bills", "attribution": src["attribution"],
            "keyword": keyword, "count": len(out), "items": out}


demo = safe_call(fetch_national_bills, "전세사기")
print(demo.get("error") or f"'{demo['keyword']}' 법률안 {demo['count']}건")
for item in sorted(demo.get("items") or [], key=lambda r: r["date"], reverse=True)[:3]:
    print("  ", item["date"], item["bill"], "|", item["proposer"], "|", item["result"])

'전세사기' 법률안 47건
   2026-09-10 전세사기피해자 지원 및 주거안정에 관한 특별법 일부개정법률안 | 염태영의원ㆍ엄태영의원 등 17인 | None
   2026-08-28 전세사기피해자 지원 및 주거안정에 관한 특별법 일부개정법률안 | 황운하의원 등 11인 | None
   2026-04-17 전세사기피해자 지원 및 주거안정에 관한 특별법 일부개정법률안 | 문진석의원 등 10인 | None


### 7-4. KOSIS 지역 통계 (17개 시도 정책 수요 점수)
이슈어와 `config/kosis_indicators.json`의 지표 키워드를 **로컬에서** 비교해 지표를 고르고,
KOSIS에서 시도별 값을 받아 순위로 0~1 '정책 수요' 점수를 만듭니다.
`backtest` 모드는 기준일 시점에 이미 공표된 연도의 값만 씁니다.

In [12]:
KOSIS_URL = "https://kosis.kr/openapi"
KOSIS_CATALOG = json.loads((BASE_DIR / "config" / "kosis_indicators.json").read_text(encoding="utf-8"))["indicators"]
DEFAULT_LAG_MONTHS = 9
TOTAL_WORDS = {"계", "합계", "전체", "총계", "소계", "전국", "총인구", "남녀전체", "남녀계", "전체연령", "합", "계(남+여)",
               "총", "종합", "총합", "전체가구", "총계(전체)"}
SHORT_REGION = {"서울": "서울특별시의회", "부산": "부산광역시의회", "대구": "대구광역시의회", "인천": "인천광역시의회",
                "광주": "광주광역시의회", "대전": "대전광역시의회", "울산": "울산광역시의회", "세종": "세종특별자치시의회",
                "경기": "경기도의회", "강원": "강원특별자치도의회", "충북": "충청북도의회", "충남": "충청남도의회",
                "전북": "전북특별자치도의회", "전남": "전라남도의회", "경북": "경상북도의회", "경남": "경상남도의회",
                "제주": "제주특별자치도의회"}


class KosisAuthError(RuntimeError):
    """인증키 오류 — 표를 바꿔 다시 시도해도 소용없으므로 즉시 중단"""


def match_indicators(keyword: str, limit: int = 4) -> list[dict]:
    """이슈어와 지표 keywords를 비교 (로컬 처리, 외부 전송 없음). 더 길게 일치한 지표를 앞에"""
    k = re.sub(r"\s+", "", keyword or "")
    out = []
    for n, ind in enumerate(KOSIS_CATALOG):
        hit = [w for w in ind["keywords"] if re.sub(r"\s+", "", w) in k or k in re.sub(r"\s+", "", w)]
        if hit:
            best = max(len(re.sub(r"\s+", "", w)) for w in hit)
            out.append((-best, -len(hit), n, {"id": ind["id"], "name": ind["name"], "category": ind["category"],
                                              "need": ind["need"], "matched": hit}))
    return [row[-1] for row in sorted(out)][:limit]


def council_of(name: str) -> str | None:
    n = re.sub(r"\s+", "", name or "")
    for council, formal in REGION.items():
        if n in formal:
            return council
    return SHORT_REGION.get(n)


def latest_visible_year(as_of: str, mode: str, lag_months: int) -> int:
    """backtest: Y년 자료는 (Y년 12월 + lag)개월에 공표된다고 보고 기준일 이전 공표분만"""
    day = date.fromisoformat(as_of[:10])
    if mode != "backtest":
        return day.year
    return (day.year * 12 + day.month - 1 - 11 - lag_months) // 12


def kosis_api(path: str, params: dict, keyword: str):
    key = api_key("KOSIS_API_KEY")
    if not key:
        raise PolicyError("KOSIS_API_KEY가 설정되지 않음")
    query = urllib.parse.urlencode({"method": "getList", "apiKey": key, "format": "json", "jsonVD": "Y", **params})
    data = json.loads(external_get(f"{KOSIS_URL}/{path}?{query}", "kosis", keyword).decode("utf-8"))
    if isinstance(data, dict) and data.get("err"):
        error = KosisAuthError if str(data.get("err")) == "11" else RuntimeError
        raise error(f"KOSIS 오류 {data.get('err')}: {data.get('errMsg')}")
    return data


def kosis_search(term: str, n: int = 10) -> list[dict]:
    term = check_keyword(term)
    hit = cache_get(f"kosis_search|{term}", max_age_days=30)
    if hit:
        return hit["value"]
    rows = kosis_api("statisticsSearch.do", {"searchNm": term, "resultCount": n}, term)
    out = [{"org_id": r.get("ORG_ID"), "org_nm": r.get("ORG_NM"), "tbl_id": r.get("TBL_ID"), "tbl_nm": r.get("TBL_NM"),
            "period": f"{r.get('STRT_PRD_DE', '')}~{r.get('END_PRD_DE', '')}"} for r in rows if r.get("TBL_ID")]
    cache_put(f"kosis_search|{term}", out)
    return out


def kosis_fetch(org_id: str, tbl_id: str, periods: int = 5) -> list[dict]:
    """통계표 최근 periods년(없으면 월별) — 분류 단계 수를 몰라 objL1..n을 늘려 가며 시도"""
    cache_key = f"kosis_data|{org_id}|{tbl_id}|{periods}"
    hit = cache_get(cache_key, max_age_days=30)
    if hit:
        return hit["value"]
    last = None
    for prd_se, n in (("Y", periods), ("M", periods * 12)):
        for depth in range(1, 5):
            params = {"orgId": org_id, "tblId": tbl_id, "itmId": "ALL", "prdSe": prd_se, "newEstPrdCnt": n,
                      **{f"objL{i}": "ALL" for i in range(1, depth + 1)}}
            try:
                rows = kosis_api("Param/statisticsParameterData.do", params, tbl_id)
                if isinstance(rows, list) and rows:
                    cache_put(cache_key, rows)
                    return rows
            except KosisAuthError:
                raise
            except RuntimeError as error:
                last = error
    raise RuntimeError(f"{tbl_id} 자료를 받지 못함: {last}")


def extract_regions(rows: list, item_hint: str | None, year_max: int) -> dict | None:
    """KOSIS 행 → {의회: 값} (지역 분류가 시도이고 나머지 분류는 '계'인 행, year_max 이하 최신 시점)"""
    items = sorted({r.get("ITM_NM", "") for r in rows})
    item = next((i for i in items if item_hint and item_hint.replace(" ", "") in i.replace(" ", "")),
                items[0] if items else "")
    cand: dict[str, dict] = {}
    for r in rows:
        if r.get("ITM_NM", "") != item:
            continue
        prd = str(r.get("PRD_DE", ""))[:4]
        if not prd.isdigit() or int(prd) > year_max:
            continue
        dims = [r.get(f"C{i}_NM") for i in range(1, 9) if r.get(f"C{i}_NM")]
        regions = [council_of(d) for d in dims]
        found = [c for c in regions if c]
        if len(found) != 1:
            continue
        if any(re.sub(r"\s+", "", d) not in TOTAL_WORDS for d, c in zip(dims, regions) if not c):
            continue
        try:
            value = float(str(r.get("DT", "")).replace(",", ""))
        except ValueError:
            continue
        cand.setdefault(prd, {}).setdefault(found[0], ([], r.get("UNIT_NM", "")))[0].append(value)  # 월별이면 연평균
    cand = {p: {c: (round(sum(vs) / len(vs), 4), u) for c, (vs, u) in d.items()} for p, d in cand.items()}
    for prd in sorted(cand, reverse=True):          # 시도 15개 이상 채워진 가장 최근 시점
        if len(cand[prd]) >= 15:
            vals = cand[prd]
            return {"period": prd, "item": item, "unit": next(iter(vals.values()))[1],
                    "values": {c: v for c, (v, _) in vals.items()}}
    return None


def need_scores(values: dict, need: str) -> dict:
    """값 순위 → 0~1 (1 = 정책 수요가 가장 큰 지역). need='high'면 큰 값, 'low'면 작은 값이 1"""
    order = sorted(values, key=lambda c: values[c], reverse=(need == "high"))
    n = max(len(order) - 1, 1)
    return {c: round(1 - i / n, 3) for i, c in enumerate(order)}


def get_regional_indicator(ind_id: str, as_of: str | None = None, mode: str = "live") -> dict:
    src = DATA_SOURCES["kosis"]
    ind = next((i for i in KOSIS_CATALOG if i["id"] == ind_id), None)
    if not ind:
        return {"error": f"없는 지표 id: {ind_id}"}
    if not key_available("kosis"):
        return {"source": src["name"], "error": f"KOSIS_API_KEY가 설정되지 않아 사용할 수 없음 ({src['terms']})"}
    as_of = as_of or date.today().isoformat()
    year_max = latest_visible_year(as_of, mode, int(ind.get("publication_lag_months") or DEFAULT_LAG_MONTHS))
    k = ind["kosis"]
    try:
        tables = ([{"org_id": k["org_id"], "tbl_id": k["tbl_id"], "tbl_nm": k.get("tbl_nm") or k["tbl_id"],
                    "org_nm": k.get("org_nm", "")}] if k.get("org_id") and k.get("tbl_id") else kosis_search(k["search"]))
    except (PolicyError, RuntimeError, ValueError, OSError) as error:
        return {"source": src["name"], "indicator": ind_id, "error": f"통계표 검색 실패: {type(error).__name__}: {error}"}
    pinned = bool(k.get("org_id") and k.get("tbl_id") and k.get("verified"))   # 사람이 확인한 표만 '확정'
    errors = []
    for t in tables[:5]:
        try:
            got = extract_regions(kosis_fetch(t["org_id"], t["tbl_id"]), k.get("item_hint"), year_max)
        except KosisAuthError as error:
            return {"source": src["name"], "indicator": ind_id, "error": f"{error} — KOSIS_API_KEY 값을 확인하세요"}
        except (RuntimeError, ValueError, OSError) as error:
            errors.append(f"{t['tbl_id']}: {type(error).__name__}: {error}")
            continue
        if not got:
            errors.append(f"{t['tbl_id']}: {year_max}년 이하에서 17개 시도 값이 나오는 행을 찾지 못함")
            continue
        return {"source": src["name"], "source_id": "kosis", "indicator": ind["id"], "name": ind["name"],
                "need": ind["need"], "table": {**t, "pinned": pinned}, "period": got["period"], "item": got["item"],
                "unit": got["unit"], "values": got["values"], "need_score": need_scores(got["values"], ind["need"]),
                "as_of": as_of, "mode": mode,
                "attribution": f"출처: 통계청 KOSIS 국가통계포털, 「{t.get('tbl_nm')}」({t.get('org_nm') or '작성기관'}; "
                               f"통계표 {t['org_id']}/{t['tbl_id']}), 수록시점 {got['period']}"
                               + ("" if pinned else " — 자동 선택된 표, 확인 필요")}
    return {"source": src["name"], "indicator": ind_id, "error": "알맞은 통계표를 찾지 못함: " + " | ".join(errors[:3])}


matched = match_indicators("출산")
print("'출산' 연결 지표:", [row["id"] for row in matched])
if matched:
    demo = safe_call(get_regional_indicator, matched[0]["id"], SETTINGS["minutes_cutoff"], "backtest")
    print(demo.get("error") or f"{demo['name']} {demo['period']} ({demo['unit']}): "
          f"수요 상위 {sorted(demo['need_score'], key=demo['need_score'].get, reverse=True)[:3]}")

'출산' 연결 지표: ['tfr', 'births']
합계출산율 2024 (가임여자 1명당 명 해당 연령 여자인구 1천 명당 명): 수요 상위 ['서울특별시의회', '부산광역시의회', '광주광역시의회']


### 7-5. 타 의회 후속 흐름 · 조례화 후보 (회의록 관측 + 외부 데이터)
의회마다 회의록 관측일과 조례 공포일을 비교해 단계(`논의_후_광역조례화`, `논의_진행_조례_미확인` 등)를 정하고,
아직 광역 조례가 없는 의회를 **미리 고정한 가중합 규칙**으로 후속 후보 순위를 매깁니다(확률·인과 아님).

| 모드 | 외부 데이터 기준일 | 회의록 |
|---|---|---|
| `backtest` (기본) | 회의록 기준일까지만, 기준일 이후 실제 제정은 사후 점검에만 사용 | 기준일까지 |
| `live` | 오늘까지 전부 | 기준일까지 |

회의록 예측은 8번 섹션 회귀 모델의 **논의 지속 확률·예상 언급량**을 씁니다(원본과 동일). 원본처럼 이 확률은
`backtest` 모드에서만 후보 점수(`issue_discussion_probability`, 가중치 0.15)에 들어가고, `live` 모드에서는 빠집니다.
회귀 예측이 없는 경우(자유검색)에는 3개월 평균 기준선으로 설명합니다.

In [13]:
ORDINANCE_KINDS = {"조례", "규칙"}
GENERIC_WORDS = {"정책", "대응", "지원", "대책", "문제", "강화", "확대", "개선", "및", "등"}
SCORING_RULE = {   # 백테스트 결과를 보고 고르지 않고 미리 고정한 가중치. 값이 None인 신호는 빼고 정규화
    "discussed_in_minutes": 0.30, "recent_discussion": 0.15, "basic_level_ordinances": 0.25,
    "regional_need": 0.15, "issue_discussion_probability": 0.15,
}
USAGE_POLICY = {"scope": "reference_only", "included_in_issue_detection": False, "included_in_forecast": False,
                "meeting_text_sent_externally": False,
                "sent_externally": "check_keyword를 통과한 짧은 정책어와 KOSIS 통계표 ID만"}
LIMITATIONS = [
    "조례·의안은 '이름'에 정책어가 들어간 것만 잡히므로 이름이 다른 관련 조례는 누락될 수 있음",
    "후속 점수는 미리 고정한 가중합 규칙이며 학습된 확률이 아님; 인과관계를 의미하지 않음",
    "회의록 관측은 제공된 2024~2025 광역의회 회의록 범위에 한정됨",
    "국회 의안은 지역 구분이 없어 점수에 넣지 않고 설명용 참고 근거로만 씀",
    "KOSIS 지역 정책 수요는 '필요가 크다'는 간접 신호이며, 자동 선택된 통계표는 확인이 필요함",
]
DIRECTION_KO = {"increase": "증가", "decrease": "감소", "stable": "유지", "unknown": "판단 불가"}


def parse_day(value: Any) -> date | None:
    try:
        return datetime.strptime(str(value or "")[:10], "%Y-%m-%d").date()
    except ValueError:
        return None


def months_before(day: date, months: int) -> date:
    index = day.year * 12 + day.month - 1 - months
    return date(index // 12, index % 12 + 1, 1)


def relevant(name: str, terms: list[str] | None) -> bool:
    """조례·의안 이름에 이슈어가 들어 있는가 (띄어쓰기 무시)"""
    if not terms:
        return True
    target = "".join(str(name or "").split())
    return any(len("".join(t.split())) >= 2 and "".join(t.split()) in target for t in terms)


def external_keywords(issue: dict, limit: int = 3) -> tuple[list[str], list[dict]]:
    """조례·의안 '이름' 검색용 짧은 정책어 (별칭 → 이슈명 조각 순서, check_keyword 통과분만)"""
    name = str(issue.get("issue_name") or "")
    candidates = [*issue.get("aliases", []), *name.split("·"), name]
    accepted, rejected = [], []
    for raw in candidates:
        keyword = " ".join(w for w in raw.split() if w not in GENERIC_WORDS).strip()
        if not keyword or keyword in accepted:
            continue
        try:
            accepted.append(check_keyword(keyword))
        except PolicyError as error:
            rejected.append({"keyword": keyword[:40], "reason": str(error)})
        if len(accepted) >= limit:
            break
    return accepted, rejected


def match_indicator_ids(terms: list[str], limit: int = 2) -> list[str]:
    """여러 이슈어에 공통으로 걸린 KOSIS 지표를 앞에"""
    hits, first_seen = {}, {}
    for term in terms:
        for row in match_indicators(term):
            hits[row["id"]] = hits.get(row["id"], 0) + 1
            first_seen.setdefault(row["id"], len(first_seen))
    return sorted(hits, key=lambda key: (-hits[key], first_seen[key]))[:limit]


def collect_external(keywords: list[str]) -> tuple[list[dict], list[dict], list[dict]]:
    """키워드별 조례·의안 조회 결과를 합친다 (시점 필터 전)"""
    ordinances, bills, errors = {}, {}, []
    for keyword in keywords:
        if key_available("law_ordin"):
            result = safe_call(fetch_local_ordinances, keyword)
            if "error" in result:
                errors.append({"source": "law_ordin", "keyword": keyword, "error": result["error"]})
            for item in result.get("items") or []:
                if item.get("kind") and item["kind"] not in ORDINANCE_KINDS:
                    continue
                identifier = item.get("law_id") or f"{item.get('org')}|{item.get('name')}"
                ordinances.setdefault(identifier, {**item, "matched_keywords": []})["matched_keywords"].append(keyword)
        if key_available("assembly_bills"):
            result = safe_call(fetch_national_bills, keyword)
            if "error" in result:
                errors.append({"source": "assembly_bills", "keyword": keyword, "error": result["error"]})
            for item in result.get("items") or []:
                bills.setdefault((str(item.get("bill")), str(item.get("date"))), item)
    return list(ordinances.values()), list(bills.values()), errors


def partition_ordinances(ordinances: list[dict], as_of: date, mode: str, terms: list[str]) -> dict:
    """모드별 시점 규칙. backtest에서 기준일 이후 개정본은 기준일 전 존재 여부를 알 수 없어 제외"""
    visible, after_enacted, unrelated, uncertain = [], [], [], 0
    for item in ordinances:
        day = parse_day(item.get("promulgated"))
        if day is None or not item.get("council"):
            continue
        if not relevant(item.get("name"), terms):
            unrelated.append(item)
        elif day <= as_of:
            visible.append(item)
        elif mode == "backtest" and item.get("type") == "제정":
            after_enacted.append(item)
        elif mode == "backtest":
            uncertain += 1
    return {"visible": visible, "after_enacted": after_enacted, "uncertain_excluded": uncertain, "unrelated": unrelated}


def regional_need(indicator_ids: list[str], as_of: date, mode: str) -> dict:
    """KOSIS 지표별 시도 수요 점수(0~1)를 의회별 평균으로"""
    if not indicator_ids or not key_available("kosis"):
        return {"by_council": {}, "indicators": [], "errors": []}
    per_council, used, errors = defaultdict(list), [], []
    for indicator_id in dict.fromkeys(indicator_ids):
        result = safe_call(get_regional_indicator, indicator_id, as_of.isoformat(), mode)
        if result.get("error"):
            errors.append({"source": "kosis", "keyword": indicator_id, "error": result["error"]})
            continue
        for council, score in result["need_score"].items():
            per_council[council].append(score)
        used.append({k: result.get(k) for k in ("indicator", "name", "need", "period", "item", "unit", "attribution")}
                    | {"pinned": (result.get("table") or {}).get("pinned")})
    return {"by_council": {c: round(sum(v) / len(v), 3) for c, v in per_council.items()},
            "indicators": used, "errors": errors}


def council_flow(propagation: list[dict], ordinances: list[dict], minutes_cutoff: date, recent_months: int) -> list[dict]:
    """의회별 회의록 관측 ↔ 조례 공포 비교로 후속 단계를 정한다."""
    observed = {normalize_council(row.get("council")): row for row in propagation if row.get("council")}
    recent_from = months_before(minutes_cutoff, recent_months)
    flows = []
    for council in REGION:
        regional = sorted((i for i in ordinances if i["council"] == council and i["level"] == "광역"),
                          key=lambda i: i["promulgated"])
        basic = sorted((i for i in ordinances if i["council"] == council and i["level"] == "기초"),
                       key=lambda i: i["promulgated"])
        minutes = observed.get(council)
        first_minutes = parse_day((minutes or {}).get("first_observation_date"))
        last_minutes = parse_day((minutes or {}).get("last_observation_date"))
        first_regional = parse_day(regional[0]["promulgated"]) if regional else None
        if minutes and first_regional:
            stage = "논의_후_광역조례화" if first_regional >= first_minutes else "광역조례_선행_후_논의"
        elif minutes:
            stage = "논의_진행_조례_미확인"
        elif regional:
            stage = "회의록_미관측_광역조례_존재"
        elif basic:
            stage = "기초조례_선행_광역논의_미관측"
        else:
            stage = "미관측"
        flows.append({
            "council": council, "stage": stage,
            "minutes_first_observation": first_minutes.isoformat() if first_minutes else None,
            "minutes_last_observation": last_minutes.isoformat() if last_minutes else None,
            "minutes_mention_count": int((minutes or {}).get("document_count") or 0),
            "recent_discussion": bool(last_minutes and last_minutes >= recent_from),
            "regional_ordinance_count": len(regional),
            "first_regional_ordinance": first_regional.isoformat() if first_regional else None,
            "regional_after_minutes_cutoff": sum(1 for i in regional if parse_day(i["promulgated"]) > minutes_cutoff),
            "lag_days_minutes_to_ordinance": ((first_regional - first_minutes).days
                                              if first_minutes and first_regional and first_regional >= first_minutes
                                              else None),
            "basic_ordinance_count": len(basic),
            "first_basic_ordinance": basic[0]["promulgated"] if basic else None,
            "regional_ordinances": [{k: i.get(k) for k in ("name", "org", "type", "promulgated", "law_id")}
                                    for i in regional[:5]],
        })
    return flows


def weighted_score(signals: dict[str, float | None]) -> float:
    available = {name: value for name, value in signals.items() if value is not None}
    total = sum(SCORING_RULE[name] for name in available)
    return sum(SCORING_RULE[n] * v for n, v in available.items()) / total if total else 0.0


def outlook(flows: list[dict], need: dict[str, float], issue_probability: float | None, top_k: int) -> dict:
    candidates = []
    for flow in flows:
        if flow["regional_ordinance_count"] > 0:
            continue   # 이미 광역 조례화 → 신규 후보가 아니라 개정·이행 점검 대상
        signals = {
            "discussed_in_minutes": 1.0 if flow["minutes_mention_count"] > 0 else 0.0,
            "recent_discussion": 1.0 if flow["recent_discussion"] else 0.0,
            "basic_level_ordinances": min(1.0, flow["basic_ordinance_count"] / 5.0),
            "regional_need": need.get(flow["council"]) if need else None,
            "issue_discussion_probability": issue_probability,
        }
        expected = ("광역_조례화" if flow["minutes_mention_count"] > 0
                    else "광역의회_논의_진입" if flow["basic_ordinance_count"] > 0 else "관망")
        candidates.append({"council": flow["council"], "expected_next_step": expected,
                           "followup_score": round(weighted_score(signals), 4), "signals": signals})
    candidates.sort(key=lambda row: (-row["followup_score"], row["council"]))
    for rank, row in enumerate(candidates, start=1):
        row["rank"] = rank
    lags = [f["lag_days_minutes_to_ordinance"] for f in flows if f["lag_days_minutes_to_ordinance"] is not None]
    return {
        "top_candidates": candidates[:top_k], "all_candidates": candidates,
        "already_institutionalized": [f["council"] for f in flows if f["regional_ordinance_count"] > 0],
        "followups_after_minutes_cutoff": [f["council"] for f in flows if f["regional_after_minutes_cutoff"] > 0],
        "median_lag_days_minutes_to_ordinance": statistics.median(lags) if lags else None,
        "scoring_rule": SCORING_RULE,
    }


def post_cutoff_check(candidates: list[dict], after_enacted: list[dict], top_k: int) -> dict:
    """backtest: 기준일 이후 실제 광역 '제정' 조례와 후보 순위를 비교"""
    enacted = sorted({i["council"] for i in after_enacted if i["level"] == "광역"})
    if not enacted:
        return {"status": "no_new_regional_enactment", "enacted_councils": []}
    top = [row["council"] for row in candidates[:top_k]]
    hits = sorted(set(top) & set(enacted))
    return {"status": "complete", "checked_until": max(i["promulgated"] for i in after_enacted),
            "enacted_councils": enacted, "top_k": top_k, "top_k_hits": hits,
            "precision_at_k": round(len(hits) / max(1, len(top)), 4), "recall_at_k": round(len(hits) / len(enacted), 4)}


def prediction_integration(forecast: dict, counts: dict, regional: dict, view: dict, mode: str) -> dict:
    """회의록 예측 숫자는 그대로 두고, 외부 신호를 후속 단계 해석의 참고로만 묶는다.
    forecast: 8번 섹션 회귀 예측(논의 지속 확률·예상 언급량). 없으면 3개월 평균 기준선으로 설명한다."""
    target = forecast.get("target_month") or "다음 달"
    probability = forecast.get("discussion_active_probability")
    mentions = forecast.get("predicted_mention_count", forecast.get("predicted_document_count"))
    institutionalized, top = list(view.get("already_institutionalized") or []), list(view.get("top_candidates") or [])
    indicators = list(regional.get("indicators") or [])
    bills, ordinances = int(counts.get("national_bills_last_12m") or 0), int(counts.get("ordinances_visible") or 0)
    lag = view.get("median_lag_days_minutes_to_ordinance")
    if probability is not None:          # 원본 external_followup.py와 같은 확률 구간 설명
        base_observation = f"{target} 논의 지속 {float(probability):.1%}" + (
            f" · 예상 언급 {int(mentions)}건" if mentions is not None else "")
        base_outlook = ("회의록 시계열은 후속 논의 지속 가능성이 높은 상태입니다." if probability >= 0.70
                        else "회의록 시계열은 제한적 지속 또는 기존 의회 중심 논의를 시사합니다." if probability >= 0.45
                        else "회의록 시계열만 보면 단기 논의 약화 가능성이 큽니다.")
        model_source, model_reflection = "회의록 시계열 모델 (8번 섹션)", "논의 지속 확률·예상 언급량"
    elif mentions is not None:
        direction = forecast.get("direction", "unknown")
        base_observation = f"{target} 예상 언급 회의록 {mentions}건 ({DIRECTION_KO.get(direction, direction)})"
        base_outlook = {"increase": "회의록 시계열은 최근 논의가 늘어나는 추세입니다.",
                        "decrease": "회의록 시계열은 최근 논의가 줄어드는 추세입니다.",
                        "stable": "회의록 시계열은 최근 논의 수준이 유지되고 있습니다."}.get(direction, "회의록 추세 판단이 어렵습니다.")
        model_source, model_reflection = "회의록 시계열 (3개월 평균 기준선)", "다음 달 예상 언급 회의록 수·증감 방향"
    else:
        base_observation, base_outlook = f"{target} 예측 없음", "회의록 예측 근거가 부족합니다."
        model_source, model_reflection = "회의록 시계열", "예측 없음"
    evidence_table = [
        {"source": model_source, "observation": base_observation,
         "reflection": model_reflection, "model_input": "사용"},
        {"source": "법제처 자치법규",
         "observation": f"관련 조례 {ordinances}건 · 광역 조례화 {len(institutionalized)}곳"
                        + (f" · 중앙 소요일 {int(lag)}일" if lag is not None else ""),
         "reflection": "후속 조례화 단계와 지역 후보 순위 보강", "model_input": "참고 신호"},
        {"source": "열린국회정보", "observation": f"최근 12개월 관련 의안 {bills}건",
         "reflection": "전국 단위 정책 모멘텀 해석", "model_input": "참고 신호"},
        {"source": "KOSIS", "observation": f"연결 지표 {len(indicators)}개"
                                           + (f" · {', '.join(str(r.get('name')) for r in indicators[:2])}" if indicators else ""),
         "reflection": "지역별 정책 수요와 후속 논의 후보 보강", "model_input": "참고 신호"},
    ]
    supporting = sum((bool(institutionalized), bills > 0, bool(indicators)))
    labels = {"광역_조례화": "광역 조례화", "광역의회_논의_진입": "광역의회 논의 진입", "관망": "관망"}
    candidate_text = (f" 후속 단계 1순위는 {top[0]['council']}"
                      f"({labels.get(top[0]['expected_next_step'], top[0]['expected_next_step'])})입니다." if top else "")
    if mode == "live":
        integrated = f"{base_outlook} 최신 외부정보는 회의록 기준 예측과 분리된 사후 참고자료입니다.{candidate_text}"
    elif supporting >= 2:
        integrated = (f"{base_outlook} 조례·국회·지역수요 가운데 {supporting}개 외부 신호가 확인되어 "
                      f"후속 논의 또는 제도화 가능성을 보강합니다.{candidate_text}")
    elif supporting == 1:
        integrated = f"{base_outlook} 외부 보강 신호는 1개로 제한적이므로 방향 해석에만 참고합니다.{candidate_text}"
    else:
        integrated = f"{base_outlook} 확인된 외부 보강 신호가 부족해 회의록 결과를 우선합니다.{candidate_text}"
    return {"official_forecast_unchanged": True, "numeric_forecast_adjusted": False,
            "external_role": "후속 단계 후보 순위와 조건부 해석 보강", "evidence_table": evidence_table,
            "integrated_outlook": integrated, "caution": "외부 참고 점수는 확률 또는 인과적 전파 판정이 아님"}


def build_followup(issue: dict, propagation: list[dict], forecast: dict, mode: str | None = None,
                   as_of: date | None = None, recent_months: int = 6, momentum_months: int = 12,
                   top_k: int = 5) -> dict:
    """회의록 관측 + 외부 공공데이터 → 후속 흐름 결과 한 건 (council_external_followup_v2 형식)"""
    mode = mode or SETTINGS["followup_mode"]
    minutes_cutoff = parse_day(SETTINGS["minutes_cutoff"])
    as_of = as_of or (date.today() if mode == "live" else minutes_cutoff)
    keywords, rejected = external_keywords(issue)
    terms = keywords + list(issue.get("aliases") or [])
    indicator_ids = match_indicator_ids(keywords + [str(issue.get("issue_name") or "")] + list(issue.get("aliases") or []))
    ordinances, bills, errors = collect_external(keywords)
    parts = partition_ordinances(ordinances, as_of, mode, terms)
    bills_visible = [b for b in bills if (d := parse_day(b.get("date"))) and d <= as_of and relevant(b.get("bill"), terms)]
    need = regional_need(indicator_ids, as_of, mode)
    errors += need["errors"]
    flows = council_flow(propagation, parts["visible"], minutes_cutoff, recent_months)
    # 원본과 같이: live에서는 회의록 기준 예측 확률(기준일 직후 1개월 대상)이 조회일 이후 전망과 맞지 않아 점수에서 뺀다
    probability = forecast.get("discussion_active_probability") if mode == "backtest" else None
    view = outlook(flows, need["by_council"], probability, top_k)
    momentum_from = months_before(as_of, momentum_months)
    counts = {
        "ordinances_visible": len(parts["visible"]), "ordinances_unrelated_excluded": len(parts["unrelated"]),
        "enacted_after_as_of": len(parts["after_enacted"]),
        "amended_after_as_of_uncertain_excluded": parts["uncertain_excluded"],
        "national_bills_visible": len(bills_visible),
        "national_bills_last_12m": sum(1 for b in bills_visible if parse_day(b.get("date")) >= momentum_from),
    }
    regional_result = {"indicators": need["indicators"], "by_council": need["by_council"]}
    used = [source for source in DATA_SOURCES if key_available(source)]
    return {
        "schema_version": "council_external_followup_v2",
        "status": "complete" if used and keywords and not errors else "partial",
        "issue": {"issue_key": issue.get("issue_key"), "issue_name": issue.get("issue_name")},
        "mode": mode, "as_of": as_of.isoformat(), "minutes_cutoff": minutes_cutoff.isoformat(),
        "outlook_horizon": f"{as_of.isoformat()} 이후" + ("" if mode == "live" else " (사후 점검 대상)"),
        "keywords": keywords, "relevance_terms": terms, "rejected_keywords": rejected, "indicators": indicator_ids,
        "usage_policy": USAGE_POLICY,
        "model_forecast_reference": {k: forecast.get(k) for k in (
            "target_month", "discussion_active_probability", "predicted_mention_count")}
                                    | {"used_in_score": probability is not None},
        "external_counts": counts,
        "followup_flow": flows,
        "regional_need": regional_result,
        "outlook": {k: v for k, v in view.items() if k != "all_candidates"},
        "prediction_integration": prediction_integration(forecast, counts, regional_result, view, mode),
        "post_cutoff_check": (post_cutoff_check(view["all_candidates"], parts["after_enacted"], top_k)
                              if mode == "backtest" else {"status": "not_applicable_live_mode"}),
        "national_bills": sorted(bills_visible, key=lambda r: r.get("date") or "", reverse=True)[:10],
        "unrelated_ordinances": [{k: i.get(k) for k in ("name", "org", "promulgated")} for i in parts["unrelated"][:10]],
        "errors": errors,
        "data_sources": sources_used(used),
        "limitations": LIMITATIONS,
    }


def compact_for_llm(result: dict, max_flows: int = 8) -> dict:
    """LLM 컨텍스트용 요약: 미관측 의회와 긴 목록은 빼고 계산된 값만"""
    view = result.get("outlook") or {}
    flows = [{k: row[k] for k in ("council", "stage", "minutes_first_observation", "minutes_mention_count",
                                  "first_regional_ordinance", "lag_days_minutes_to_ordinance", "basic_ordinance_count")}
             for row in result.get("followup_flow") or [] if row.get("stage") != "미관측"][:max_flows]
    return {
        "mode": result.get("mode"), "as_of": result.get("as_of"), "minutes_cutoff": result.get("minutes_cutoff"),
        "keywords": result.get("keywords"), "followup_flow": flows,
        "already_institutionalized": view.get("already_institutionalized"),
        "top_candidates": [{k: row[k] for k in ("rank", "council", "expected_next_step", "followup_score")}
                           for row in view.get("top_candidates") or []],
        "median_lag_days_minutes_to_ordinance": view.get("median_lag_days_minutes_to_ordinance"),
        "regional_need_indicators": [{k: row.get(k) for k in ("name", "period", "pinned")}
                                     for row in (result.get("regional_need") or {}).get("indicators") or []],
        "national_bills_last_12m": (result.get("external_counts") or {}).get("national_bills_last_12m"),
        "recent_national_bills": [{k: row.get(k) for k in ("bill", "date", "result")}
                                  for row in (result.get("national_bills") or [])[:3]],
        "integrated_outlook": (result.get("prediction_integration") or {}).get("integrated_outlook"),
        "errors": [row.get("error") for row in result.get("errors") or []][:3],
        "attribution": [row.get("attribution") for row in result.get("data_sources") or []],
        "note": "점수는 미리 고정한 규칙 기반 참고값이며 인과관계·확률이 아님. 회의록 원문은 외부로 전송되지 않음",
    }


EXTERNAL_ENABLED = any(key_available(source) for source in DATA_SOURCES)
print("외부 공공데이터 사용:", EXTERNAL_ENABLED, "| 모드:", SETTINGS["followup_mode"])

외부 공공데이터 사용: True | 모드: live


## 8. 향후 예측: 다중 이슈 회귀 모델 (Poisson)
원본 `team-ggeodeureok-radar/experiments/kwonjun_chroma/src`의 예측 파이프라인을 옮겼습니다.

| 단계 | 원본 | 내용 |
|---|---|---|
| 8-1 | `20_expand_cluster_issue.py` | 이슈 검색 상위 10건을 seed로, seed 평균 벡터 주변 500개 청크를 확장 → 중심 유사도 ≥ 0.45 → 중복 제거 → 유사도 0.72 그래프 군집 중 seed가 든 군집 = **이슈 구성 청크(member)** |
| 8-2 | `22_build_forecast_features.py` | member의 월×의회 언급 수 → **누수 없는 월별 특징 19개**(특징은 그 달까지, 정답은 다음 달) |
| 8-3 | `31_build_multi_issue_panel.py` | 대표 이슈 10개의 특징을 하나의 패널로 결합 |
| 8-4 | `32_train_multi_issue_forecast.py` | 원본의 시간순 검증(rolling backtest)에서 **최종 선택된 방법**으로 고정: 다음 달 논의 지속 확률 = 최근 3개월 활동 비율, 다음 달 언급량 = **PoissonRegressor** → 전체 기간으로 학습해 기준일 다음 달 예측 |

원본 검증 결과(참고, `multi_issue_forecast.json`): 논의 지속 F1 0.88, 언급량 MAE 7.17건, 방향성 정확도 0.77.
모델·방법 비교와 검증 코드는 이 노트북에서 뺐습니다(원본 32번 스크립트에 있음).

원본과 다른 점: seed가 BM25+BGE 재순위 대신 이 노트북의 하이브리드 검색(5번) 상위 10건이고, 사람 검수 단계가 없습니다.
'언급'을 전문검색 일치 수로 세면 '정책'·'대응' 같은 일반어 때문에 모든 달이 '논의 중'이 되어 학습이 안 되므로,
원본처럼 **의미 군집 member 수**를 씁니다(9번 이슈 분석 화면의 월별 언급 수와는 단위가 다름). member는 `data/forecast/`에 캐시합니다.

### 8-1. 이슈 구성 청크(member): seed 확장 · 그래프 군집

In [14]:
import numpy as np

FORECAST_DIR = BASE_DIR / "data" / "forecast"
LOOKBACK, LONG_LOOKBACK, TOTAL_COUNCILS = 3, 6, 17          # 원본 22번 기본값
TARGET_COLUMNS = ("next_month_mention_count", "target_discussion_active",
                  "target_discussion_growth", "target_new_council_spread")
ID_COLUMNS = ("issue_id", "issue_name", "feature_month", "target_month")
CUTOFF_MONTH = SETTINGS["minutes_cutoff"][:7]


SEED_COUNT, EXPANSION_K, EF_SEARCH = 10, 500, 500           # 원본 20번 기본값
MIN_CENTROID_SIMILARITY, CLUSTER_SIMILARITY, MIN_CLUSTER_SIZE = 0.45, 0.72, 5


def _to_vector(value: Any) -> np.ndarray:
    value = value.to_numpy() if hasattr(value, "to_numpy") else value
    return np.asarray(value, dtype=np.float32)


def graph_clusters(vectors: np.ndarray, threshold: float, block_size: int = 128) -> np.ndarray:
    """코사인 유사도 ≥ threshold인 청크끼리 잇고 연결요소(Union-Find)를 군집으로"""
    parent = list(range(len(vectors)))

    def find(i: int) -> int:
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    for start in range(0, len(vectors), block_size):
        similarities = vectors[start:start + block_size] @ vectors.T
        for local, row in enumerate(similarities):
            i = start + local
            for j in np.flatnonzero(row >= threshold):
                if j > i:
                    parent[find(i)] = find(int(j))
    roots = [find(i) for i in range(len(vectors))]
    labels = {root: n for n, root in enumerate(dict.fromkeys(roots))}
    return np.asarray([labels[r] for r in roots])


def issue_members(issue: dict) -> list[dict[str, Any]]:
    """원본 20번: 대표 이슈를 구성하는 회의록 청크(member) 목록. 결과는 캐시"""
    cache_key = hashlib.md5(f"{TABLE}|{issue['query']}".encode()).hexdigest()
    cache_file = FORECAST_DIR / f"members_{cache_key}.json"
    if cache_file.exists():
        return json.loads(cache_file.read_text(encoding="utf-8"))
    seed_ids = [row["chunk_id"] for row in search(issue["query"], top_k=SEED_COUNT)["results"]]
    with connect() as conn:
        seed_vectors = {r["chunk_id"]: _to_vector(r["embedding"]) for r in conn.execute(
            f"SELECT chunk_id, embedding FROM {TABLE} WHERE chunk_id = ANY(%s)", (seed_ids,)).fetchall()}
        centroid = np.stack([seed_vectors[s] for s in seed_ids]).mean(axis=0)
        centroid = centroid / np.linalg.norm(centroid)
        conn.execute("SELECT set_config('hnsw.ef_search', %s, true)", (str(EF_SEARCH),))
        conn.execute("SELECT set_config('hnsw.iterative_scan', 'strict_order', true)")
        rows = conn.execute(f"""
            SELECT chunk_id, parent_chunk_id, council, meeting_date, document, embedding,
                   1 - (embedding <=> %s) AS similarity
            FROM {TABLE} ORDER BY embedding <=> %s LIMIT %s""", (centroid, centroid, EXPANSION_K)).fetchall()
        found = {r["chunk_id"] for r in rows}
        missing = [s for s in seed_ids if s not in found]
        if missing:   # 확장 범위 밖 seed도 포함
            rows += conn.execute(f"""
                SELECT chunk_id, parent_chunk_id, council, meeting_date, document, embedding,
                       1 - (embedding <=> %s) AS similarity
                FROM {TABLE} WHERE chunk_id = ANY(%s)""", (centroid, missing)).fetchall()
    seeds = set(seed_ids)
    candidates = [r for r in rows if r["similarity"] >= MIN_CENTROID_SIMILARITY or r["chunk_id"] in seeds]
    # 같은 부모 청크·같은 원문(공백 무시)은 하나만 남긴다 (seed 우선, 다음은 유사도 높은 것)
    for key_of in (lambda r: r["parent_chunk_id"] or r["chunk_id"],
                   lambda r: hashlib.sha256(re.sub(r"\s+", "", r["document"] or "").casefold().encode()).hexdigest()):
        best: dict[str, dict] = {}
        for r in candidates:
            k, prev = key_of(r), best.get(key_of(r))
            if prev is None or (r["chunk_id"] in seeds, r["similarity"]) > (prev["chunk_id"] in seeds, prev["similarity"]):
                best[k] = r
        candidates = list(best.values())
    if len(candidates) < MIN_CLUSTER_SIZE:
        raise RuntimeError(f"{issue['issue_key']}: 중심 유사도 조건을 통과한 후보가 너무 적습니다 ({len(candidates)}개)")
    vectors = np.stack([_to_vector(r["embedding"]) for r in candidates])
    vectors = vectors / np.maximum(np.linalg.norm(vectors, axis=1, keepdims=True), 1e-12)
    labels = graph_clusters(vectors, CLUSTER_SIMILARITY)
    seed_labels = {labels[i] for i, r in enumerate(candidates) if r["chunk_id"] in seeds}
    if not seed_labels:
        seed_labels = {np.bincount(labels).argmax()}
    members = [{"chunk_id": r["chunk_id"], "council": normalize_council(r["council"] or ""),
                "meeting_date": _iso(r["meeting_date"])[:10] if r["meeting_date"] else None,
                "centroid_similarity": round(float(r["similarity"]), 6)}
               for i, r in enumerate(candidates) if labels[i] in seed_labels]
    FORECAST_DIR.mkdir(parents=True, exist_ok=True)
    cache_file.write_text(json.dumps(members, ensure_ascii=False), encoding="utf-8")
    return members


def _iso(value: Any) -> str:
    return value.isoformat() if isinstance(value, (date, datetime)) else str(value)

### 8-2. member → 월별 예측 특징 (누수 방지)

In [15]:
def add_month(month: str, amount: int = 1) -> str:
    year, number = map(int, month.split("-"))
    total = year * 12 + number - 1 + amount
    return f"{total // 12:04d}-{total % 12 + 1:02d}"


def _slope(values: list[float]) -> float:
    return float(np.polyfit(np.arange(len(values), dtype=np.float64), np.asarray(values, dtype=np.float64), 1)[0]) \
        if len(values) >= 2 else 0.0


def _padded_tail(values: list, end: int, size: int) -> list[float]:
    selected = values[max(0, end - size + 1): end + 1]
    return [0.0] * (size - len(selected)) + [float(v) for v in selected]


def _months_since_positive(values: list[int], end: int) -> int:
    for distance, value in enumerate(reversed(values[: end + 1])):
        if value > 0:
            return distance
    return end + 1


def _concentration(counts: dict[str, int]) -> tuple[float, float, float]:
    """의회 집중도: HHI, 최대 점유율, 정규화 엔트로피"""
    total = sum(counts.values())
    if total <= 0:
        return 0.0, 0.0, 0.0
    shares = np.asarray([v / total for v in counts.values()], dtype=np.float64)
    entropy = 0.0 if len(shares) <= 1 else float(-(shares * np.log(shares)).sum() / np.log(len(shares)))
    return float(np.square(shares).sum()), float(shares.max()), entropy


def build_issue_features(issue: dict, members: list[dict]) -> list[dict[str, Any]]:
    """원본 22번: 관측 없는 달은 0으로 채우고, 각 행의 특징은 feature_month까지·정답은 다음 달에서만 계산"""
    counts: dict[str, dict[str, int]] = defaultdict(lambda: defaultdict(int))
    similarities: dict[str, list[float]] = defaultdict(list)
    for m in members:
        if m["meeting_date"] and m["council"] and m["meeting_date"][:7] <= CUTOFF_MONTH:
            counts[m["meeting_date"][:7]][m["council"]] += 1
            similarities[m["meeting_date"][:7]].append(m["centroid_similarity"])
    observed = sorted(m for m in counts if counts[m])
    if not observed:
        return []
    months, current = [], observed[0]
    while current <= CUTOFF_MONTH:
        months.append(current)
        current = add_month(current)
    first_month_by_council: dict[str, str] = {}
    for month in months:
        for council in counts.get(month, {}):
            first_month_by_council.setdefault(council, month)
    mentions = [sum(counts.get(m, {}).values()) for m in months]
    active = [len(counts.get(m, {})) for m in months]
    new_counts = [sum(first == m for first in first_month_by_council.values()) for m in months]
    cumulative = list(np.cumsum(new_counts))
    rows = []
    for index in range(LONG_LOOKBACK - 1, len(months)):
        recent = _padded_tail(mentions, index, LOOKBACK)
        previous = _padded_tail(mentions, index - LOOKBACK, LOOKBACK) if index >= LOOKBACK else [0.0] * LOOKBACK
        recent_mean, previous_mean = float(np.mean(recent)), float(np.mean(previous))
        hhi, top_share, entropy = _concentration(counts.get(months[index], {}))
        next_known = index + 1 < len(months)
        next_mentions = mentions[index + 1] if next_known else None
        rows.append({
            "issue_id": issue["issue_key"], "issue_name": issue["issue_name"],
            "feature_month": months[index], "target_month": add_month(months[index]),
            "mention_count": mentions[index],
            "active_council_count": active[index],
            "new_council_count": new_counts[index],
            "cumulative_council_count": int(cumulative[index]),
            "council_coverage_ratio": round(int(cumulative[index]) / TOTAL_COUNCILS, 6),
            "mention_mean_3m": round(recent_mean, 6),
            "mention_mean_previous_3m": round(previous_mean, 6),
            "mention_momentum_3m": round((recent_mean - previous_mean) / max(previous_mean, 1.0), 6),
            "mention_slope_3m": round(_slope(recent), 6),
            "mention_slope_6m": round(_slope(_padded_tail(mentions, index, LONG_LOOKBACK)), 6),
            "active_council_mean_3m": round(float(np.mean(_padded_tail(active, index, LOOKBACK))), 6),
            "new_council_sum_3m": int(sum(_padded_tail(new_counts, index, LOOKBACK))),
            "active_month_ratio_3m": round(sum(v > 0 for v in recent) / LOOKBACK, 6),
            "months_since_last_mention": _months_since_positive(mentions, index),
            "issue_age_months": index + 1,
            "council_hhi": round(hhi, 6), "top_council_share": round(top_share, 6), "council_entropy": round(entropy, 6),
            "centroid_similarity_mean": round(float(np.mean(similarities[months[index]])), 6)
                                        if similarities.get(months[index]) else 0.0,
            "next_month_mention_count": next_mentions if next_known else None,
            "target_discussion_active": int(next_mentions > 0) if next_known else None,
            "target_discussion_growth": int(next_mentions > recent_mean and next_mentions > 0) if next_known else None,
            "target_new_council_spread": int(new_counts[index + 1] > 0) if next_known else None,
        })
    return rows


started = time.perf_counter()
ISSUE_MEMBERS: dict[str, list[dict]] = {}
FEATURE_ROWS: dict[str, list[dict]] = {}
for issue in ISSUE_LIST:
    ISSUE_MEMBERS[issue["issue_key"]] = issue_members(issue)
    FEATURE_ROWS[issue["issue_key"]] = rows = build_issue_features(issue, ISSUE_MEMBERS[issue["issue_key"]])
    active_months = sum(r["mention_count"] > 0 for r in rows)
    print(f"{issue['issue_key']:20} member {len(ISSUE_MEMBERS[issue['issue_key']]):3} | 특징 행 {len(rows):2} "
          f"(학습 가능 {sum(r['target_discussion_active'] is not None for r in rows)}, 언급 있는 달 {active_months}) "
          f"| 최근 {rows[-1]['feature_month'] if rows else '-'} 언급 {rows[-1]['mention_count'] if rows else '-'}")
print(f"member 구성·특징 생성 {time.perf_counter() - started:.0f}초")

birth_rate           member 115 | 특징 행 19 (학습 가능 18, 언급 있는 달 14) | 최근 2025-12 언급 8


population_decline   member 163 | 특징 행 19 (학습 가능 18, 언급 있는 달 14) | 최근 2025-12 언급 8
youth_jobs           member 126 | 특징 행 19 (학습 가능 18, 언급 있는 달 13) | 최근 2025-12 언급 3
jeonse_fraud         member 139 | 특징 행 19 (학습 가능 18, 언급 있는 달 15) | 최근 2025-12 언급 11
regional_healthcare  member  19 | 특징 행 16 (학습 가능 15, 언급 있는 달 6) | 최근 2025-12 언급 0
care_policy          member 103 | 특징 행 19 (학습 가능 18, 언급 있는 달 12) | 최근 2025-12 언급 2
school_violence      member  40 | 특징 행 19 (학습 가능 18, 언급 있는 달 14) | 최근 2025-12 언급 1
climate_neutrality   member 176 | 특징 행 19 (학습 가능 18, 언급 있는 달 13) | 최근 2025-12 언급 14
flood_safety         member  28 | 특징 행 14 (학습 가능 13, 언급 있는 달 6) | 최근 2025-12 언급 0
small_business       member 224 | 특징 행 19 (학습 가능 18, 언급 있는 달 17) | 최근 2025-12 언급 13
member 구성·특징 생성 0초


### 8-3. 다중 이슈 패널 → 8-4. 다음 달 예측 (원본 최종 선택 방법으로 고정)

In [16]:
import joblib
from sklearn.compose import TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import PoissonRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 원본 32번의 시간순 검증에서 최종 선택된 방법 (multi_issue_forecast.json의 selection)
SELECTED_CLASSIFIER = "active_ratio_baseline"     # 논의 지속 확률 = 최근 3개월 중 언급 있는 달 비율 (0.02~0.98로 제한)
SELECTED_COUNT_MODEL = "poisson_regression"       # 다음 달 언급량 = PoissonRegressor(log1p 변환)

# ---- 8-3. 패널: 이슈별 특징을 합치고 (target_month, issue_id) 순으로 정렬 (원본 31번)
PANEL = sorted((row for rows in FEATURE_ROWS.values() for row in rows),
               key=lambda r: (r["target_month"], r["issue_id"]))
if len({(r["issue_id"], r["feature_month"]) for r in PANEL}) != len(PANEL):
    raise ValueError("issue_id×feature_month 중복")
FEATURE_COLUMNS = [c for c in PANEL[0] if c not in (*ID_COLUMNS, *TARGET_COLUMNS)]
LABELED = [r for r in PANEL if all(r[c] is not None for c in TARGET_COLUMNS)]   # 다음 달 정답이 있는 행
LATEST = [max(rows, key=lambda r: r["feature_month"]) for rows in FEATURE_ROWS.values() if rows]   # 기준일이 속한 달


def to_matrix(rows: list[dict]) -> np.ndarray:
    return np.asarray([[float(r[c]) for c in FEATURE_COLUMNS] for r in rows], dtype=np.float64)


def regression_pipeline() -> TransformedTargetRegressor:
    pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),
                         ("model", PoissonRegressor(alpha=0.5, max_iter=3000))])
    return TransformedTargetRegressor(regressor=pipeline, func=np.log1p, inverse_func=np.expm1)


def change_direction(current: float, future: float) -> str:
    """언급 건수를 정수 단위로 비교해 증가·지속·감소"""
    now, later = int(round(max(0.0, current))), int(round(max(0.0, future)))
    return "growth" if later > now else "attenuation" if later < now else "persistence"


# ---- 8-4. 전체 학습 행으로 언급량 모델 학습 → 각 이슈의 마지막 특징 행으로 다음 달 예측
regressor = regression_pipeline().fit(to_matrix(LABELED),
                                      np.asarray([float(r["next_month_mention_count"]) for r in LABELED]))
FORECASTS: dict[str, dict[str, Any]] = {}
for r, count in zip(LATEST, np.maximum(0.0, regressor.predict(to_matrix(LATEST)))):
    probability = min(0.98, max(0.02, float(r["active_month_ratio_3m"])))
    FORECASTS[r["issue_id"]] = {
        "issue_id": r["issue_id"], "issue_name": r["issue_name"],
        "feature_month": r["feature_month"], "target_month": r["target_month"],
        "current_mention_count": int(r["mention_count"]),
        "discussion_active_probability": round(probability, 6),
        "predicted_mention_count": int(round(float(count))),
        "direction": change_direction(float(r["mention_count"]), float(count)),
        "selected_classifier": SELECTED_CLASSIFIER, "selected_count_model": SELECTED_COUNT_MODEL,
    }

FORECAST_REPORT = {
    "status": "complete",
    "method": "원본 32번 시간순 검증의 최종 선택 방법으로 고정 (이 노트북에서는 재검증하지 않음)",
    "leakage_control": "특징은 feature_month까지, 정답은 다음 달에서만 계산",
    "count_unit": "이슈 의미 군집 member 청크 수 (원본과 동일 단위)",
    "issue_count": len(FEATURE_ROWS), "training_rows": len(LABELED), "feature_columns": FEATURE_COLUMNS,
    "selection": {"classifier": SELECTED_CLASSIFIER, "count_model": SELECTED_COUNT_MODEL},
    "forecasts": list(FORECASTS.values()),
    "limitations": ["대표 이슈 수가 적어 확률 보정 전의 실험 모델임",
                    "예측은 제공된 2024~2025 회의록 이력만 사용함",
                    "이슈 정의와 검색어 품질 변화에 민감하므로 사람 검수가 선행되어야 함"],
}
FORECAST_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(regressor, FORECAST_DIR / "poisson_regression.joblib")
(FORECAST_DIR / "multi_issue_forecast.json").write_text(json.dumps(FORECAST_REPORT, ensure_ascii=False, indent=2),
                                                         encoding="utf-8")

print(f"이슈 {FORECAST_REPORT['issue_count']} | 학습 행 {len(LABELED)} | 방법 {FORECAST_REPORT['selection']}")
print("\n다음 달 예측:")
for f in sorted(FORECASTS.values(), key=lambda f: -f["discussion_active_probability"]):
    print(f"  {f['target_month']} {f['issue_name']:22} 논의 지속 {f['discussion_active_probability']:.0%} | "
          f"예상 언급 {f['predicted_mention_count']}건 (현재 {f['current_mention_count']}건, {f['direction']})")

이슈 10 | 학습 행 172 | 방법 {'classifier': 'active_ratio_baseline', 'count_model': 'poisson_regression'}

다음 달 예측:
  2026-01 저출생·출산율 대응 정책          논의 지속 98% | 예상 언급 2건 (현재 8건, attenuation)
  2026-01 인구감소·지역소멸 대응           논의 지속 98% | 예상 언급 3건 (현재 8건, attenuation)
  2026-01 청년 일자리와 지역정착 지원        논의 지속 98% | 예상 언급 3건 (현재 3건, persistence)
  2026-01 기후위기와 탄소중립 전환          논의 지속 98% | 예상 언급 3건 (현재 14건, attenuation)
  2026-01 소상공인 경영 안정 지원          논의 지속 98% | 예상 언급 3건 (현재 13건, attenuation)
  2026-01 전세사기 피해 예방과 지원         논의 지속 67% | 예상 언급 2건 (현재 11건, attenuation)
  2026-01 돌봄 공백과 통합돌봄 정책         논의 지속 67% | 예상 언급 6건 (현재 2건, growth)
  2026-01 학교폭력 예방과 피해자 보호        논의 지속 67% | 예상 언급 2건 (현재 1건, growth)
  2026-01 지역의료 공백과 필수의료 대응       논의 지속 33% | 예상 언급 2건 (현재 0건, growth)
  2026-01 집중호우·침수 재난 대응          논의 지속 33% | 예상 언급 2건 (현재 0건, growth)


## 9. 이슈 분석: 월별 언급 · 의회별 최초 관측 · 다음 달 예측 · 외부 후속 흐름
- 월별 언급: 전문검색에 걸린 **서로 다른 회의록(document_id) 수**
- 의회별 최초 관측: 의회마다 처음 언급된 날짜 (시간 순서일 뿐 인과적 전파가 아님)
- `forecast`: 최근 3개월 평균 기준선 (main.py·대시보드 예측 카드와 동일)
- `multi_issue_forecast`: 8번 섹션 예측의 다음 달 **논의 지속 확률·예상 언급량**과 사용한 방법 (대표 이슈만)
- 외부 후속 흐름: 7번 섹션 `build_followup` 결과를 `external_followup`에 붙임 (키가 하나도 없으면 생략).
  회귀 예측 확률은 원본처럼 `backtest` 모드에서만 후속 후보 점수에 들어갑니다.
- `followup_report`: 위 결과를 **타 지방의회 후속 논의 흐름 및 향후 예측** 항목별 서술형 보고서로 정리 (9-2 셀)
- 같은 이슈는 노트북이 켜져 있는 동안 결과를 재사용(`ANALYSIS_CACHE`)합니다

In [17]:
def _iso(value: Any) -> str:
    return value.isoformat() if isinstance(value, (date, datetime)) else str(value)


def monthly_counts(query: str, filters: dict | None = None) -> list[dict[str, Any]]:
    where_sql, where_values = where_clause(filters)
    with connect() as conn:
        rows = conn.execute(f"""
            WITH q AS (SELECT websearch_to_tsquery('simple', %s) AS term)
            SELECT date_trunc('month', meeting_date)::date AS month,
                   min(meeting_date)::date AS first_observation_date,
                   count(DISTINCT document_id)::bigint AS document_count,
                   count(DISTINCT council)::bigint AS council_count,
                   array_agg(DISTINCT council) FILTER (WHERE council IS NOT NULL) AS councils
            FROM {TABLE}, q
            WHERE meeting_date IS NOT NULL
              AND to_tsvector('simple', coalesce(document, '')) @@ q.term{where_sql}
            GROUP BY 1 ORDER BY 1""", [full_text_query(query), *where_values]).fetchall()
    return [{"month": _iso(r["month"]), "first_observation_date": _iso(r["first_observation_date"]),
             "document_count": int(r["document_count"]), "council_count": int(r["council_count"]),
             "councils": r["councils"] or []} for r in rows]


def council_first_observations(query: str, filters: dict | None = None) -> list[dict[str, Any]]:
    where_sql, where_values = where_clause(filters)
    with connect() as conn:
        rows = conn.execute(f"""
            WITH q AS (SELECT websearch_to_tsquery('simple', %s) AS term)
            SELECT council, min(meeting_date)::date AS first_observation_date,
                   max(meeting_date)::date AS last_observation_date,
                   count(DISTINCT document_id)::bigint AS document_count
            FROM {TABLE}, q
            WHERE meeting_date IS NOT NULL AND council IS NOT NULL
              AND to_tsvector('simple', coalesce(document, '')) @@ q.term{where_sql}
            GROUP BY council ORDER BY first_observation_date, council""",
            [full_text_query(query), *where_values]).fetchall()
    return [{"council": r["council"], "first_observation_date": _iso(r["first_observation_date"]),
             "last_observation_date": _iso(r["last_observation_date"]),
             "document_count": int(r["document_count"])} for r in rows]


def _next_month(d: date) -> date:
    return date(d.year + (d.month == 12), 1 if d.month == 12 else d.month + 1, 1)


def forecast_monthly_counts(rows: list[dict[str, Any]]) -> dict[str, Any]:
    """최근 3개월 평균으로 다음 달 언급 회의록 수를 예측하는 투명한 기준선."""
    counts = {date.fromisoformat(r["month"][:10]).replace(day=1): r["document_count"] for r in rows}
    base = {"method": "trailing_3_month_mean", "direction": "unknown", "predicted_document_count": None}
    if not counts:
        return {**base, "status": "insufficient_history", "target_month": None, "history_months": 0}
    monthly, cursor = [], min(counts)
    while cursor <= max(counts):
        monthly.append(counts.get(cursor, 0))
        cursor = _next_month(cursor)
    target = _next_month(max(counts)).strftime("%Y-%m")
    if len(monthly) < 3:
        return {**base, "status": "insufficient_history", "target_month": target, "history_months": len(monthly)}
    recent = mean(monthly[-3:])
    previous = mean(monthly[-6:-3]) if len(monthly) >= 6 else None
    direction = ("stable" if previous is None or abs(recent - previous) < 1e-9
                 else "increase" if recent > previous else "decrease")
    return {**base, "status": "complete", "target_month": target,
            "predicted_document_count": max(0, int(round(recent))), "direction": direction,
            "history_months": len(monthly), "recent_3_month_mean": round(recent, 3),
            "previous_3_month_mean": round(previous, 3) if previous is not None else None}


def keyword_snippet(text: str, query: str, width: int = 240) -> str:
    """가장 구체적인(긴) 검색어가 처음 나오는 곳 주변을 잘라 카드에 보여 준다."""
    for term in sorted(set(re.findall(r"[0-9A-Za-z가-힣]{2,}", query)), key=len, reverse=True):
        at = text.find(term)
        if at >= 0:
            start = max(0, at - 60)
            return ("…" if start else "") + text[start:start + width] + ("…" if start + width < len(text) else "")
    return text[:width]


def council_representatives(query: str, councils: list[str], near_k: int = 1000) -> dict[str, dict[str, Any]]:
    """의회별 대표 발언 1건. 카드의 날짜·문서 수와 같은 전문검색 매칭 청크 중에서만 고른다
    (매칭 날짜가 있는 의회는 대표 발언도 반드시 있도록).
    1) 질의와 가까운 청크 near_k개 중 의회마다 매칭 청크 하나  2) 없는 의회는 그 의회 매칭 청크 중 가장 가까운 것."""
    vector = embed_query(query)
    terms = full_text_query(query)
    with connect() as conn:
        conn.execute("SELECT set_config('hnsw.ef_search', %s, true)", (str(min(1000, near_k)),))
        rows = conn.execute(f"""
            WITH q AS (SELECT websearch_to_tsquery('simple', %s) AS term),
            near AS (SELECT {FIELDS}, 1 - (embedding <=> %s::vector) AS similarity FROM {TABLE}
                     WHERE embedding IS NOT NULL ORDER BY embedding <=> %s::vector, chunk_id LIMIT %s)
            SELECT DISTINCT ON (council) near.* FROM near, q
            WHERE council IS NOT NULL AND meeting_date IS NOT NULL
              AND to_tsvector('simple', coalesce(document, '')) @@ q.term
            ORDER BY council, similarity DESC""",
            [terms, vector, vector, near_k]).fetchall()
        found = {r["council"]: dict(r) for r in rows}
        for council in [c for c in councils if c not in found]:
            row = conn.execute(f"""
                WITH q AS (SELECT websearch_to_tsquery('simple', %s) AS term)
                SELECT {FIELDS}, 1 - (embedding <=> %s::vector) AS similarity FROM {TABLE}, q
                WHERE council = %s AND meeting_date IS NOT NULL AND embedding IS NOT NULL
                  AND to_tsvector('simple', coalesce(document, '')) @@ q.term
                ORDER BY (embedding <=> %s::vector) + 0, chunk_id LIMIT 1""",
                [terms, vector, council, vector]).fetchone()
            if row:
                found[council] = dict(row)
    out = {}
    for council in councils:
        if council in found:
            row = public_row(found[council])
            row.update({"similarity": found[council].get("similarity"),
                        "snippet": keyword_snippet(row["text"], query)})
            out[council] = row
    return out


def build_analysis(query: str, issue: dict | None = None, top_k: int = 10) -> dict[str, Any]:
    found = search(query, top_k=top_k, candidate_k=max(50, top_k * 5))
    monthly = monthly_counts(query)
    propagation = council_first_observations(query)
    forecast = {**forecast_monthly_counts(monthly),
                "basis": "postgresql_full_text_matches", "count_unit": "distinct_document_id"}
    # 최초 관측 의회 = 가장 이른 관측일에 언급한 의회들 (첫 달 전체 의회가 아님)
    first_date = propagation[0]["first_observation_date"] if propagation else None
    first_councils = [r["council"] for r in propagation if r["first_observation_date"] == first_date]
    model_forecast = None
    if issue and issue["issue_key"] in globals().get("FORECASTS", {}):     # 8번 섹션 회귀 예측 (대표 이슈만)
        model_forecast = {**FORECASTS[issue["issue_key"]], **{k: FORECAST_REPORT[k] for k in (
            "selection", "method", "leakage_control",
            "count_unit", "limitations")}}
    followup, followup_status = None, "not_configured"
    if EXTERNAL_ENABLED and issue:      # 외부 조회는 검수된 대표 이슈에만 (자유검색어를 외부로 보내지 않음)
        followup = build_followup(issue, propagation, model_forecast or forecast)
        followup_status = followup["status"]
    result = {
        "schema_version": "council_issue_db_v1",
        "status": "complete" if monthly else "no_matching_history",
        "issue": {
            "issue_id": issue["issue_key"] if issue else hashlib.sha256(query.encode()).hexdigest()[:12],
            "issue_key": issue["issue_key"] if issue else None,
            "issue_name": issue["issue_name"] if issue else query,
            "aliases": issue["aliases"] if issue else [],
            "status": "fixed" if issue else "provisional",
        },
        "query": query,
        "summary": {
            "first_observation_date": first_date,
            "first_observation_councils": first_councils,
            "council_count": len({r["council"] for r in propagation}),
            "evidence_unit_count": sum(r["document_count"] for r in monthly),
            "analysis_scope": "postgresql_minutes_table",
        },
        "monthly_counts": monthly,
        "forecast": forecast,
        "multi_issue_forecast": model_forecast,
        "evidence": found["results"],
        "retrieval": {"backend": found["backend"], "mode": found["retrieval_mode"],
                      "result_count": found["result_count"], "search_ms": found["search_ms"]},
        "propagation": propagation,
        # 의회별 대표 발언 (관측 의회 카드용): propagation과 같은 매칭 조건이라 모든 관측 의회에 1건씩 있다
        "council_evidence": council_representatives(query, [r["council"] for r in propagation]),
        "external_followup": followup,
        "external_followup_status": followup_status,
    }
    result["followup_report"] = followup_report(result)     # 9-2 셀: 항목별 서술 보고서
    return result


ANALYSIS_CACHE: dict[str, dict[str, Any]] = {}


def get_issue(issue_key: str) -> dict[str, Any] | None:
    issue = ISSUE_BY_KEY.get(issue_key)
    if issue is None:
        return None
    if issue_key not in ANALYSIS_CACHE:
        ANALYSIS_CACHE[issue_key] = build_analysis(issue["query"], issue)
    return ANALYSIS_CACHE[issue_key]


def radar_search(query: str) -> dict[str, Any]:
    """자유검색: 대표 이슈로 연결되면 그 이슈를, 아니면 검색어 그대로 동적 분석한다."""
    issue, score = route_fixed_issue(query)
    if issue:
        result = dict(get_issue(issue["issue_key"]))
        result.update({"match_type": "fixed_issue", "matched_issue_id": issue["issue_key"]})
    else:
        result = build_analysis(query)
        result.update({"match_type": "dynamic", "matched_issue_id": None})
        if result["status"] == "complete":
            result["status"] = "provisional_until_review"
    result.update({"routing_score": score, "query": query})
    return result

### 9-2. 타 지방의회 후속 논의 흐름 및 향후 예측 (항목별 서술)
9번 분석 결과(회의록 관측·8번 예측·7번 외부 후속 흐름)를 **코드가 정해진 문장 규칙으로** 항목별 서술형 보고서로 만듭니다.
LLM을 쓰지 않으므로 숫자·의회명은 계산 결과 그대로이고, 문장 규칙은 원본 Streamlit 화면(`demo/final_app.py`)의
단계 이름·확률 구간별 전망 문장을 따릅니다. 결과는 `followup_report`(API 응답, 대시보드 패널)에 담깁니다.

In [18]:
FOLLOWUP_STAGE_LABELS = {      # 원본 demo/final_app.py
    "논의_후_광역조례화": "논의 → 광역 조례화",
    "광역조례_선행_후_논의": "광역 조례 선행 → 논의",
    "논의_진행_조례_미확인": "논의 중 · 조례 미확인",
    "회의록_미관측_광역조례_존재": "회의록 미관측 · 광역 조례 있음",
    "기초조례_선행_광역논의_미관측": "기초 조례 선행 · 광역 논의 없음",
}
NEXT_STEP_LABELS = {"광역_조례화": "광역 조례화", "광역의회_논의_진입": "광역의회 논의 진입", "관망": "관망"}
FORECAST_DIRECTION_LABELS = {"growth": "논의 증가", "persistence": "논의 지속", "attenuation": "관심 약화"}


def forecast_outlook_sentence(issue_name: str, probability: float) -> str:
    """원본 forecast_interpretation의 확률 구간별 조건부 전망 문장"""
    if probability >= 0.70:
        return (f"‘{issue_name}’ 관련 사업 집행 상황, 예산 반영 여부와 정책 효과를 점검하는 "
                "후속 논의가 계속될 가능성이 높습니다.")
    if probability >= 0.45:
        return (f"‘{issue_name}’ 논의가 전체 지역으로 빠르게 확산되기보다는 기존 관측 의회를 중심으로 "
                "후속 질의와 집행 점검 논의가 이어질 가능성이 있습니다.")
    return (f"‘{issue_name}’의 새로운 지역 확산보다는 기존 안건의 정리와 사업 점검 중심으로 "
            "논의 강도가 낮아질 가능성이 있습니다.")


def _short(council: str) -> str:
    return council.removesuffix("의회")


def followup_report(payload: dict[str, Any]) -> dict[str, Any]:
    """타 지방의회 후속 논의 흐름과 향후 예측을 항목별 서술 문장으로 정리한다."""
    issue_name = payload["issue"]["issue_name"]
    summary, propagation = payload["summary"], payload["propagation"]
    followup, model = payload.get("external_followup"), payload.get("multi_issue_forecast")
    sections: list[dict[str, Any]] = []

    # 1. 회의록에서 관측된 논의 확산 흐름
    items = []
    if propagation:
        items.append(f"{summary['first_observation_date']}에 {', '.join(summary['first_observation_councils'])}에서 "
                     f"처음 관측된 뒤, 모두 {summary['council_count']}개 광역의회에서 관련 논의가 확인되었습니다.")
        order = " → ".join(f"{_short(r['council'])}({r['first_observation_date']})" for r in propagation[:8])
        items.append(f"의회별 최초 관측 순서: {order}"
                     + (f" 외 {len(propagation) - 8}곳" if len(propagation) > 8 else "")
                     + ". 이 순서는 시간적 관측 순서이며 의회 간 인과적 전파를 뜻하지 않습니다.")
        recent_from = months_before(parse_day(SETTINGS["minutes_cutoff"]), 6).isoformat()
        recent = [_short(r["council"]) for r in propagation if (r.get("last_observation_date") or "") >= recent_from]
        if recent:
            items.append(f"회의록 기준일 직전 6개월 안에도 논의가 이어진 의회는 {len(recent)}곳입니다: {', '.join(recent)}.")
    else:
        items.append("회의록에서 관련 논의가 관측되지 않았습니다.")
    sections.append({"title": "회의록에서 관측된 논의 확산 흐름", "items": items})

    # 2. 타 지방의회 후속 논의 단계 (회의록 관측 + 법제처 조례)
    items = []
    if followup:
        flows = [f for f in followup["followup_flow"] if f["stage"] != "미관측"]
        for stage, label in FOLLOWUP_STAGE_LABELS.items():
            rows = [f for f in flows if f["stage"] == stage]
            if not rows:
                continue
            if stage == "논의_후_광역조례화":
                detail = ", ".join(f"{_short(f['council'])}(첫 논의 {f['minutes_first_observation']} → 조례 "
                                   f"{f['first_regional_ordinance']}, {f['lag_days_minutes_to_ordinance']}일)" for f in rows)
            elif stage == "광역조례_선행_후_논의":
                detail = ", ".join(f"{_short(f['council'])}(조례 {f['first_regional_ordinance']} → 첫 논의 "
                                   f"{f['minutes_first_observation']})" for f in rows)
            elif stage == "논의_진행_조례_미확인":
                detail = ", ".join(f"{_short(f['council'])}(첫 논의 {f['minutes_first_observation']}"
                                   + (f", 관할 기초 조례 {f['basic_ordinance_count']}건" if f["basic_ordinance_count"] else "")
                                   + ")" for f in rows)
            else:
                detail = ", ".join(f"{_short(f['council'])}(기초 조례 {f['basic_ordinance_count']}건)" for f in rows)
            items.append(f"{label} ({len(rows)}곳): {detail}.")
        lag = followup["outlook"].get("median_lag_days_minutes_to_ordinance")
        if lag is not None:
            items.append(f"회의록 첫 논의에서 광역 조례 공포까지 걸린 기간의 중앙값은 {int(lag)}일입니다.")
        after = followup["outlook"].get("followups_after_minutes_cutoff") or []
        if after:
            items.append(f"회의록 기준일({followup['minutes_cutoff']}) 이후 실제로 광역 조례가 공포된 의회: "
                         f"{', '.join(_short(c) for c in after)}.")
    else:
        items.append("외부 공공데이터(법제처 자치법규)가 연결되지 않아 조례화 단계는 판단하지 않았습니다.")
    sections.append({"title": "타 지방의회 후속 논의 단계", "items": items})

    # 3. 향후 후속 조례화·논의 진입 후보 (규칙 기반 참고 점수)
    items = []
    if followup and followup["outlook"]["top_candidates"]:
        for c in followup["outlook"]["top_candidates"]:
            s = c["signals"]
            reasons = ["회의록 논의 있음" if s["discussed_in_minutes"] else "회의록 논의 없음"]
            if s["recent_discussion"]:
                reasons.append("최근 6개월 논의")
            flow = next(f for f in followup["followup_flow"] if f["council"] == c["council"])
            if flow["basic_ordinance_count"]:
                reasons.append(f"관할 기초 조례 {flow['basic_ordinance_count']}건")
            if s.get("regional_need") is not None:
                reasons.append(f"지역 정책 수요 {s['regional_need']:.2f}")
            items.append(f"{c['rank']}순위 {c['council']}: 예상 다음 단계 "
                         f"‘{NEXT_STEP_LABELS.get(c['expected_next_step'], c['expected_next_step'])}’ "
                         f"(참고 점수 {c['followup_score']:.2f} · {', '.join(reasons)}).")
        institutionalized = followup["outlook"]["already_institutionalized"]
        if institutionalized:
            items.append(f"이미 광역 조례가 있는 {len(institutionalized)}곳({', '.join(_short(c) for c in institutionalized)})은 "
                         "신규 조례화보다 개정·이행 점검 대상으로 보아 후보에서 제외했습니다.")
        items.append("참고 점수는 미리 정한 가중합 규칙의 값이며 확률이나 인과관계가 아닙니다.")
    else:
        items.append("후속 후보를 계산할 외부 데이터가 없습니다.")
    sections.append({"title": "향후 후속 조례화·논의 진입 후보", "items": items})

    # 4. 다음 달 논의 전망 (8번 예측)
    items = []
    if model:
        probability, predicted = model["discussion_active_probability"], model["predicted_mention_count"]
        items.append(f"{model['target_month']}에도 논의가 이어질 확률 {probability:.1%} · "
                     f"예상 언급 {predicted}건 (최근 달 {model['current_mention_count']}건, "
                     f"{FORECAST_DIRECTION_LABELS.get(model['direction'], model['direction'])}).")
        items.append(forecast_outlook_sentence(issue_name, probability))
        latest = max(FEATURE_ROWS.get(payload["issue"]["issue_key"]) or [{}], key=lambda r: r.get("feature_month", ""))
        if latest:
            items.append(f"근거 신호: 최근 관측월 언급 {latest['mention_count']}건 · 3개월 평균 "
                         f"{latest['mention_mean_3m']:.1f}건 · 활성 의회 {latest['active_council_count']}개 · "
                         f"누적 관측 의회 {latest['cumulative_council_count']}개.")
        items.append("논의 지속 확률은 최근 3개월 활동 비율, 예상 언급량은 Poisson 회귀 값이며(원본 최종 선택 방법), "
                     "언급 단위는 이슈 의미 군집의 회의록 청크 수입니다.")
    else:
        forecast = payload["forecast"]
        if forecast.get("predicted_document_count") is not None:
            items.append(f"{forecast['target_month']} 예상 매칭 회의록 {forecast['predicted_document_count']}건 "
                         f"(최근 3개월 평균 기준선, 방향 {DIRECTION_KO.get(forecast['direction'], '판단 불가')}).")
            items.append({"increase": "유사 정책 제안과 후속 점검 논의가 늘어날 가능성이 있습니다.",
                          "stable": "기존 관측 의회를 중심으로 관련 논의가 이어질 가능성이 있습니다.",
                          "decrease": "새로운 확산보다는 기존 논의의 정리·점검이 중심이 될 가능성이 있습니다."
                          }.get(forecast["direction"], "현재 근거만으로 논의 방향을 단정하기 어렵습니다."))
        else:
            items.append("향후 논의 방향을 계산할 근거가 부족합니다.")
    sections.append({"title": "다음 달 논의 전망", "items": items})

    # 5. 외부 근거를 반영한 종합 해석
    if followup:
        items = [followup["prediction_integration"]["integrated_outlook"]]
        bills = followup.get("national_bills") or []
        count_12m = followup["external_counts"].get("national_bills_last_12m", 0)
        if bills:
            recent_bill = bills[0]
            items.append(f"국회에서는 최근 12개월간 관련 법률안 {count_12m}건이 발의되었습니다"
                         f"(최근: {recent_bill['date']} 「{recent_bill['bill']}」).")
        for row in followup["regional_need"]["indicators"]:
            need = followup["regional_need"]["by_council"]
            top_need = sorted(need, key=need.get, reverse=True)[:3]
            items.append(f"지역 정책 수요 지표 ‘{row['name']}’({row['period']}) 기준 수요가 큰 지역: "
                         f"{', '.join(_short(c) for c in top_need)}.")
        sections.append({"title": "외부 근거를 반영한 종합 해석", "items": items})

    # 6. 유의사항·출처
    items = ["회의록 근거는 제공된 2024~2025년 광역의회 회의록 범위에 한정됩니다."]
    if followup:
        items.append(f"외부 데이터는 {followup['mode']} 모드, 기준일 {followup['as_of']}의 참고 정보이며 "
                     "회의록 기반 예측 수치를 바꾸지 않습니다.")
        items += [row["attribution"] for row in followup.get("data_sources") or []]
        items += [row["attribution"] for row in followup["regional_need"]["indicators"]]
    sections.append({"title": "유의사항 · 출처", "items": items})

    text = "\n".join(f"{n}. {s['title']}\n" + "\n".join(f"  - {item}" for item in s["items"])
                     for n, s in enumerate(sections, start=1))
    return {"title": f"‘{issue_name}’ 타 지방의회 후속 논의 흐름 및 향후 예측", "sections": sections, "text": text}

이슈 분석 실행 예시 (전문검색 인덱스가 없으면 수 분 걸립니다):

In [19]:
started = time.perf_counter()
analysis = get_issue("jeonse_fraud")
print(f"분석 {time.perf_counter() - started:.0f}초 | 상태 {analysis['status']}")
print("최초 관측:", analysis["summary"]["first_observation_date"], analysis["summary"]["first_observation_councils"])
print("관측 의회 수:", analysis["summary"]["council_count"], "| 언급 회의록 수:", analysis["summary"]["evidence_unit_count"])
print("기준선 예측:", {k: analysis["forecast"][k] for k in ("target_month", "predicted_document_count", "direction")})
model_forecast = analysis["multi_issue_forecast"]
if model_forecast:
    print(f"회귀 예측: {model_forecast['target_month']} 논의 지속 {model_forecast['discussion_active_probability']:.0%}, "
          f"예상 언급 {model_forecast['predicted_mention_count']}건 ({model_forecast['selection']})")
print("의회별 최초 관측 (앞 5개):")
for row in analysis["propagation"][:5]:
    print("  ", row["first_observation_date"], row["council"], f"{row['document_count']}건")

followup = analysis["external_followup"]
if followup:
    print(f"\n[외부 후속 흐름 · {followup['mode']} · 기준 {followup['as_of']} · {followup['status']}]")
    print("검색어:", followup["keywords"], "| KOSIS 지표:", followup["indicators"])
    print("외부 건수:", followup["external_counts"])
    print("광역 조례화 의회:", followup["outlook"]["already_institutionalized"])
    print("후속 후보:", [(c["council"], c["expected_next_step"], c["followup_score"])
                       for c in followup["outlook"]["top_candidates"]])
    print("통합 해석:", followup["prediction_integration"]["integrated_outlook"])
    for error in followup["errors"]:
        print("  오류:", error)

report = analysis["followup_report"]
print()
print("=" * 80)
print(report["title"])
print("=" * 80)
print(report["text"])

분석 71초 | 상태 complete
최초 관측: 2024-01-08 ['서울특별시의회']
관측 의회 수: 17 | 언급 회의록 수: 5552
기준선 예측: {'target_month': '2026-01', 'predicted_document_count': 362, 'direction': 'increase'}
회귀 예측: 2026-01 논의 지속 67%, 예상 언급 2건 ({'classifier': 'active_ratio_baseline', 'count_model': 'poisson_regression'})
의회별 최초 관측 (앞 5개):
   2024-01-08 서울특별시의회 590건
   2024-01-18 인천광역시의회 324건
   2024-01-23 전라남도의회 351건
   2024-01-23 충청남도의회 423건
   2024-01-23 충청북도의회 278건

[외부 후속 흐름 · live · 기준 2026-11-03 · complete]
검색어: ['전세 피해', '임차인 보호', '주거 피해'] | KOSIS 지표: ['jeonse_index']
외부 건수: {'ordinances_visible': 45, 'ordinances_unrelated_excluded': 37, 'enacted_after_as_of': 0, 'amended_after_as_of_uncertain_excluded': 0, 'national_bills_visible': 2, 'national_bills_last_12m': 1}
광역 조례화 의회: ['서울특별시의회', '부산광역시의회', '인천광역시의회', '대전광역시의회', '세종특별자치시의회', '경기도의회', '강원특별자치도의회', '충청남도의회', '전북특별자치도의회', '경상남도의회', '제주특별자치도의회']
후속 후보: [('울산광역시의회', '광역_조례화', 0.7426), ('대구광역시의회', '광역_조례화', 0.6912), ('광주광역시의회', '광역_조례화', 0.6286), ('충청북도의회', '광역

## 10. 원문 근거·회의록 조회

In [20]:
def get_evidence(chunk_id: str) -> dict[str, Any] | None:
    with connect() as conn:
        row = conn.execute(f"SELECT {FIELDS} FROM {TABLE} WHERE chunk_id = %s LIMIT 1", (chunk_id,)).fetchone()
    return public_row(dict(row)) if row else None


def get_document(document_id: str, limit: int = 100) -> dict[str, Any] | None:
    with connect() as conn:
        rows = conn.execute(f"SELECT {FIELDS} FROM {TABLE} WHERE document_id = %s "
                            "ORDER BY meeting_date, chunk_id LIMIT %s", (document_id, limit)).fetchall()
    if not rows:
        return None
    return {"document_id": document_id, "result_count": len(rows), "items": [public_row(dict(r)) for r in rows]}


first_hit = analysis["evidence"][0]
evidence = get_evidence(first_hit["chunk_id"])
document = get_document(first_hit["document_id"], limit=500)
print("근거 청크:", evidence["chunk_id"], "|", evidence["metadata"].get("council"), evidence["metadata"].get("meeting_date"))
print("같은 회의록의 청크 수:", document["result_count"])

근거 청크: chunk_30d2ac50a472495a1668276ff7388c09 | 전라남도의회 2025-03-11
같은 회의록의 청크 수: 74


## 11. 로컬 LLM 채팅 (RAG + 이슈 분석 + 예측 + 외부 공공데이터)
1. **검색창과 같은 근거를 먼저** 붙입니다. 질문이 대표 이슈로 연결되면 검색창의 이슈 대표 근거 10건(9번 캐시)을
   같은 순서로 `[근거 1~10]`에 넣고, 질문에 이슈 이름·별칭 밖의 단어가 있을 때만(예: "조례를 만들 지역은?")
   질문 문장으로 검색한 근거 중 겹치지 않는 것을 최대 4건 뒤에 덧붙입니다. "전세사기"처럼 이슈 단어만 넣으면 검색창과 똑같은 10건입니다.
   자유 질문은 검색창 동적 분석과 같은 조건(상위 10건·후보 50건)으로 검색하므로 단어 하나를 넣으면
   검색창 근거 목록과 똑같은 근거가 됩니다. 근거 수에 따라 청크를 1,800/1,000/700자로 줄여 컨텍스트(8192)에 맞춥니다.
2. 질문이 대표 이슈로 연결되면(6번 섹션) 9번 섹션의 **이슈 분석 결과**(최초 관측·관측 순서·8번 회귀 예측)와
   7번 섹션의 **외부 공공데이터 요약**을 `[이슈 분석 결과]`, `[외부 공공데이터]`로 함께 넣습니다.
   연결되지 않는 자유 질문은 회의록 근거만 사용합니다(main.py와 동일).
3. 시스템 프롬프트로 출처 표기·숫자 복사·인과 해석 금지 규칙을 지시합니다.
4. 외부 공공데이터를 프롬프트에 넣은 경우, LLM 답변과 상관없이 **코드가 답변 끝에 출처 블록을 붙이고**
   응답 `external_sources`에도 같은 출처 목록을 담습니다. 대시보드 채팅창은 답변 본문을 그대로 보여 주므로
   화면 코드를 고치지 않아도 출처가 표시됩니다.

질문의 원문은 외부로 나가지 않습니다(외부 조회는 대표 이슈의 검수된 별칭만 사용).

In [21]:
CHAT_SYSTEM_PROMPT = (
    "지방의회 회의록 근거를 사용해 한국어로 답하세요. 근거에 없는 사실이나 숫자는 만들지 말고, "
    "각 사실 뒤에 [근거 N]을 표시하세요. 근거가 부족하면 그렇게 말하세요."
)
CHAT_ANALYSIS_RULES = (
    " [이슈 분석 결과]와 [외부 공공데이터]가 주어지면 그 값도 사용할 수 있습니다. "
    "숫자·날짜·의회명은 주어진 값을 그대로 옮기고, 이슈 분석 값은 [분석], 외부 값은 "
    "[외부: 법제처]·[외부: 국회]·[외부: KOSIS]로 표시하세요. "
    "의회 간 최초 관측 순서는 인과적 전파가 아닌 시간적 관측 순서이고, 논의 지속 확률은 보정 전 실험 모델의 값이며, "
    "후속 후보 점수는 미리 정한 규칙의 참고값이지 확률이 아니라고 밝히세요. "
    "외부 데이터의 기준일(as_of)과 회의록 기준일을 한 번 언급하세요."
)


def external_sources(followup: dict) -> list[dict[str, Any]]:
    """채팅에 넣은 외부 데이터의 출처 목록 (어떤 출처에서 무엇을 썼는지, 기준일 포함)"""
    counts = followup.get("external_counts") or {}
    used = {row["id"]: row for row in followup.get("data_sources") or []}
    common = {"mode": followup.get("mode"), "as_of": followup.get("as_of"),
              "minutes_cutoff": followup.get("minutes_cutoff"), "keywords": followup.get("keywords")}
    sources = []
    if "law_ordin" in used:
        sources.append({"source_id": "law_ordin", **common, "attribution": used["law_ordin"]["attribution"],
                        "terms": used["law_ordin"]["terms"],
                        "used_data": f"관련 자치법규 {counts.get('ordinances_visible', 0)}건 (공포일 기준일 이하)"})
    if "assembly_bills" in used:
        sources.append({"source_id": "assembly_bills", **common, "attribution": used["assembly_bills"]["attribution"],
                        "terms": used["assembly_bills"]["terms"],
                        "used_data": f"관련 국회의원 발의 법률안 {counts.get('national_bills_visible', 0)}건 "
                                     f"(최근 12개월 {counts.get('national_bills_last_12m', 0)}건)"})
    if "kosis" in used:
        indicators = (followup.get("regional_need") or {}).get("indicators") or []
        for row in indicators or [{}]:      # 지표별 통계표·수록시점까지 남긴다
            sources.append({"source_id": "kosis", **common,
                            "attribution": row.get("attribution") or used["kosis"]["attribution"],
                            "terms": used["kosis"]["terms"],
                            "used_data": f"지역별 정책 수요 지표: {row['name']} ({row.get('period')})" if row
                                         else "연결된 지표 없음"})
    return sources


def format_sources(sources: list[dict[str, Any]]) -> str:
    head = sources[0]
    lines = [f"[외부 공공데이터 출처 · {head['mode']} 모드 · 외부 데이터 기준일 {head['as_of']} · "
             f"회의록 기준일 {head['minutes_cutoff']} · 참고용]"]
    lines += [f"- {row['attribution']} — {row['used_data']}" for row in sources]
    lines.append(f"- 외부 조회 검색어: {', '.join(head['keywords'] or [])} (회의록 원문·질문은 외부로 보내지 않음)")
    lines.append("※ 후속 후보 점수는 미리 정한 규칙의 참고값이며 확률·인과관계가 아닙니다.")
    return "\n".join(lines)


def issue_context(issue_key: str) -> tuple[str, list[dict[str, Any]]]:
    """대표 이슈 분석 결과를 LLM이 읽을 짧은 텍스트로 만든다 (계산은 코드가, 설명은 LLM이).
    외부 데이터를 넣었으면 그 출처 목록도 함께 돌려준다."""
    payload = get_issue(issue_key)
    summary, forecast = payload["summary"], payload["forecast"]
    order = " → ".join(f"{r['council']}({r['first_observation_date']})" for r in payload["propagation"][:8])
    recent = ", ".join(f"{r['month'][:7]}: {r['document_count']}건" for r in payload["monthly_counts"][-6:])
    lines = [
        f"이슈: {payload['issue']['issue_name']} (데이터: 2024~2025년 광역의회 회의록)",
        f"최초 관측: {summary['first_observation_date']} / {', '.join(summary['first_observation_councils'][:5])}",
        f"관측 의회 수: {summary['council_count']}곳, 언급 회의록 수: {summary['evidence_unit_count']}건",
        f"시간적 관측 순서(인과 아님): {order}",
        f"최근 월별 언급 회의록 수: {recent}",
    ]
    model = payload.get("multi_issue_forecast")
    if model:   # 8번 섹션 예측이 있으면 그것만 넘긴다 (단위가 다른 기준선과 섞이지 않게)
        lines.append(
            f"다음 달 예측(8번 섹션: 논의 지속 = 최근 3개월 활동 비율, 언급량 = Poisson 회귀): "
            f"{model['target_month']} 논의 지속 확률 {model['discussion_active_probability']:.1%}, "
            f"이슈 군집 언급 {model['predicted_mention_count']}건 예상 (최근 달 {model['current_mention_count']}건)")
    else:
        lines.append(
            f"다음 달 기준선({forecast['method']}): {forecast.get('target_month')} 예상 "
            f"{forecast.get('predicted_document_count')}건, 방향 {DIRECTION_KO.get(forecast.get('direction'), '판단 불가')}")
    text = "[이슈 분석 결과]\n" + "\n".join(lines)
    sources: list[dict[str, Any]] = []
    if payload.get("external_followup"):
        text += "\n\n[외부 공공데이터]\n" + json.dumps(compact_for_llm(payload["external_followup"]), ensure_ascii=False)
        sources = external_sources(payload["external_followup"])
    return text, sources


SEARCH_BOX_TOP_K = 10        # 검색창(build_analysis) 근거 개수와 같게
CHAT_EXTRA_EVIDENCE = 4      # 대표 이슈 질문에서 검색창 근거 뒤에 덧붙일 질문 근거 최대 개수


def merge_evidence(base_rows: list[dict], extra_rows: list[dict], extra_limit: int) -> list[dict]:
    """검색창 근거를 같은 순서로 먼저 두고, 질문 근거 중 겹치지 않는 것만 뒤에 덧붙인다."""
    merged = {row["chunk_id"]: {**row, "evidence_basis": ["search_box"]} for row in base_rows}
    added = 0
    for row in extra_rows:
        if row["chunk_id"] in merged:
            merged[row["chunk_id"]]["evidence_basis"].append("question")
        elif added < extra_limit:
            merged[row["chunk_id"]] = {**row, "evidence_basis": ["question"]}
            added += 1
    return list(merged.values())


def question_extra_terms(message: str, issue: dict) -> set[str]:
    """이슈 이름·표준 질의·별칭에 없는 질문 단어 (조사 붙은 형태 '전세사기는'도 같은 단어로 본다)."""
    known = _terms(" ".join([issue["issue_name"], issue["query"], *issue["aliases"]]))
    return {t for t in _terms(message) if not any(t.startswith(k) for k in known)}


def chat(message: str, history: list[dict[str, str]] | None = None, top_k: int = 6,
         conversation_id: str | None = None) -> dict[str, Any]:
    url = SETTINGS["ollama_url"].rstrip("/")
    if (urlparse(url).hostname or "").lower() not in LOCAL_HOSTS:
        raise RuntimeError("보안 정책상 localhost Ollama만 사용할 수 있습니다")
    issue, routing_score = route_fixed_issue(message)
    context, sources, warning, issue_rows = "", [], None, []
    if issue:
        try:
            # 검색창과 같은 캐시: 이슈 표준 질의로 찾은 대표 근거 10건 (추가 검색 없음)
            issue_rows = get_issue(issue["issue_key"])["evidence"]
            context, sources = issue_context(issue["issue_key"])
            context += "\n\n"
        except Exception as error:   # 분석 실패해도 회의록 근거만으로 답한다
            warning = f"이슈 분석 생략: {type(error).__name__}: {error}"
    extra_terms = question_extra_terms(message, issue) if issue_rows else set()
    if issue_rows and extra_terms:   # [근거 1~10] = 검색창 근거, 그 뒤에 질문 문장 근거 중 새로운 것
        found = search(message, top_k=top_k, candidate_k=max(50, top_k * 5))
        evidence = merge_evidence(issue_rows, found["results"], CHAT_EXTRA_EVIDENCE)
    elif issue_rows:    # 이슈 단어만 있는 질문(예: '전세사기'): 검색창 근거 10건 그대로, 추가 검색 없음
        retrieval = get_issue(issue["issue_key"])["retrieval"]
        found = {"backend": retrieval["backend"], "retrieval_mode": retrieval["mode"], "results": [],
                 "query_embedding_ms": 0.0, "search_ms": 0.0}
        evidence = merge_evidence(issue_rows, [], 0)
    else:               # 자유 질문: 검색창 동적 분석과 같은 조건(상위 10건, 후보 50건)으로 검색
        k = max(top_k, SEARCH_BOX_TOP_K)
        found = search(message, top_k=k, candidate_k=max(50, k * 5))
        evidence = merge_evidence(found["results"], [], 0)
    if not evidence:
        raise RuntimeError("PostgreSQL 검색 결과가 없어 답변 근거를 찾지 못했습니다")
    # 근거가 많을수록 청크를 줄여 컨텍스트(num_ctx)에 맞춘다
    text_limit = 1800 if len(evidence) <= 6 else (1000 if len(evidence) <= 10 else 700)
    citations, blocks = [], []
    for index, row in enumerate(evidence, start=1):
        m = row["metadata"]
        citations.append({"citation": index, "chunk_id": row["chunk_id"], "council": m.get("council"),
                          "meeting_date": m.get("meeting_date"), "meeting_name": m.get("meeting_name"),
                          "agenda_title": m.get("agenda_title"), "text": row["text"],
                          "evidence_basis": row["evidence_basis"]})
        blocks.append(f"[근거 {index}] {m.get('council', '미상')} | {m.get('meeting_date', '날짜 미상')}\n"
                      f"{row['text'][:text_limit]}")
    system = CHAT_SYSTEM_PROMPT + (CHAT_ANALYSIS_RULES if context else "")
    messages = [{"role": "system", "content": system}, *(history or []),
                {"role": "user", "content": f"질문: {message}\n\n{context}PostgreSQL 검색 근거:\n" + "\n\n".join(blocks)}]
    body = json.dumps({"model": SETTINGS["chat_model"], "stream": False, "messages": messages,
                       "options": {"num_ctx": SETTINGS["chat_num_ctx"]}}, ensure_ascii=False).encode("utf-8")
    request = urllib.request.Request(url + "/api/chat", data=body,
                                     headers={"Content-Type": "application/json"}, method="POST")
    try:
        with urllib.request.urlopen(request, timeout=SETTINGS["chat_timeout"]) as response:
            payload = json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as error:     # Ollama가 돌려준 오류 이유(예: 컨텍스트 초과)를 함께 보여 준다
        detail = error.read().decode("utf-8", errors="replace")[:300]
        raise RuntimeError(f"로컬 Ollama 답변 생성 실패: HTTP {error.code}: {detail}") from error
    except OSError as error:
        raise RuntimeError(f"로컬 Ollama 답변 생성 실패: {type(error).__name__}: {error}") from error
    answer = str((payload.get("message") or {}).get("content") or "").strip()
    if not answer:
        raise RuntimeError("Ollama가 빈 답변을 반환했습니다")
    if sources:   # LLM이 출처를 빠뜨리거나 잘못 써도 항상 남도록 코드가 붙인다
        answer += "\n\n" + format_sources(sources)
    return {
        "answer": answer, "citations": citations, "external_sources": sources,
        "conversation_id": conversation_id or uuid4().hex[:24],
        "retrieval_mode": found["retrieval_mode"], "generation_mode": f"ollama:{SETTINGS['chat_model']}",
        "warning": warning,
        "plan": {"backend": found["backend"], "result_count": len(evidence),
                 "evidence_basis": {"search_box": sum("search_box" in r["evidence_basis"] for r in evidence),
                                    "question": sum("question" in r["evidence_basis"] for r in evidence)},
                 "query_embedding_ms": found["query_embedding_ms"],
                 "matched_issue_id": issue["issue_key"] if issue else None, "routing_score": routing_score,
                 "question_extra_terms": sorted(extra_terms),
                 "issue_analysis_used": bool(context), "external_data_used": bool(sources)},
        "policy": {"external_llm_api_used": False, "meeting_text_sent_externally": False,
                   "question_text_sent_externally": False, "answer_grounded_in_retrieved_evidence": True},
        "search_ms": found["search_ms"],
    }

채팅 실행 예시:

In [22]:
for question in ["전세사기 논의가 다음 달에도 이어질까? 언급량은 어느 정도로 예상돼?",
                 "전세사기 문제는 어느 의회에서 먼저 나왔고, 앞으로 어느 지역이 조례를 만들 가능성이 높아?"]:
    started = time.perf_counter()
    reply = chat(question)
    plan = reply["plan"]
    print(f"Q. {question}")
    print(f"[{reply['generation_mode']}] {time.perf_counter() - started:.0f}초 | 이슈 {plan['matched_issue_id']} | "
          f"분석 {plan['issue_analysis_used']} | 외부 {plan['external_data_used']}\n")
    print(reply["answer"])
    print("\n인용:", [f"{c['citation']}:{c['council']} {c['meeting_date']}" for c in reply["citations"]])
    print("external_sources:", [row["source_id"] for row in reply["external_sources"]])
    print("=" * 80)

# 대표 이슈로 연결되지 않는 질문 → 외부 데이터를 쓰지 않으므로 출처 블록도 없음
reply = chat("드론 배송 규제에 대해 논의된 내용은?")
print("자유 질문 | 외부 사용:", reply["plan"]["external_data_used"], "| 출처 블록:",
      "[외부 공공데이터 출처" in reply["answer"])

Q. 전세사기 논의가 다음 달에도 이어질까? 언급량은 어느 정도로 예상돼?
[ollama:hf.co/mykor/A.X-4.0-Light-gguf:Q4_K_M] 44초 | 이슈 jeonse_fraud | 분석 True | 외부 True

전세사기 논의가 다음 달에도 이어질 가능성은 높습니다. [이슈 분석 결과]에 따르면, 전세사기 피해 예방과 지원에 대한 논의는 2024년 1월부터 지속되어 왔으며, 특히 2025년 11월까지의 월별 언급량이 꾸준히 증가하고 있습니다. 2025년 10월에는 146건, 11월에는 659건으로 크게 증가했으며, 12월에는 280건으로 여전히 높은 수준을 유지하고 있습니다. [분석]에 따르면, 2026년 1월의 논의 지속 확률은 66.7%로 예측되었으며, 이는 해당 이슈가 당분간 지속될 가능성이 있음을 시사합니다.

**언급량 예상**
- 2026년 1월의 이슈 군집 언급 예상 건수: 2건 (최근 달 11건 기준)
- 이는 Poisson 회귀를 통한 예측 결과로, 최근 3개월 간의 활동 비율을 반영한 것입니다.

**추가적 고려 사항**
- 전세사기 피해자 수는 계속 증가하고 있으며, 2023년 6월부터 시행된 특별법의 유효 기간이 2025년 5월까지로 한정되어 있어, 이에 대한 연장 논의가 필수적입니다. [외부: 법제처]의 데이터에 따르면, 전세사기 피해자 지원 및 주거 안정에 관한 특별법의 유효 기간 연장은 주요 논의 주제 중 하나입니다.
- 서울특별시의 경우, 2024년 한 해 동안 25개 구에서 41건의 지원이 이루어졌으나, 이는 전체 가결 건수 7,900건과 비교하면 현격히 낮은 수치로, 추가적인 지원이 필요하다는 점이 강조되고 있습니다. [근거 3, 4, 5, 6, 7, 8]

**결론**
전세사기 문제는 여전히 심각한 사회적 이슈로 남아 있으며, 다음 달에도 관련 논의가 계속될 것으로 보입니다. 특히, 피해자 보호 및 지원 체계 강화, 특별법 유효 기간 연장 등이 주요 쟁점으로 다뤄질 것입니다. **다음 달에도 관련 회의나

Q. 전세사기 문제는 어느 의회에서 먼저 나왔고, 앞으로 어느 지역이 조례를 만들 가능성이 높아?
[ollama:hf.co/mykor/A.X-4.0-Light-gguf:Q4_K_M] 45초 | 이슈 jeonse_fraud | 분석 True | 외부 True

**전세사기 문제 최초 발생 및 미래 조례 가능성 분석**

**1. 전세사기 문제의 최초 발생**
- **최초 발생 의회**: 서울특별시의회 [근거 1]
- **최초 관측일**: 2024-01-08
- **최초 언급 회의록 수**: 590건
- **시간적 관측 순서**: 서울특별시의회 → 인천광역시의회 → 전라남도의회 → 충청남도의회 → 충청북도 → 대전광역시 → 대구광역시 → 부산광역시 [근거 1]

**2. 조례 제정 가능성이 높은 지역**
- **분석 결과**: 울산광역시의회가 가장 높은 가능성을 가짐 [외부: 법제처]
- **이유**:
  - **울산광역시의회**는 2024-02-15에 처음으로 전세사기 문제가 논의되었으며, 이후 논의 후 광역 조례화가 진행될 가능성이 높다고 평가됨 [외부: 법제처].
  - **예상 다음 단계**: 광역 조례화 [외부: 법제처].
  - **후속 후보 점수**: 0.7426으로 가장 높은 점수를 기록 [외부: 법제처].

- **기타 가능성 있는 지역**:
  - **대구광역시**: 논의 진행 중이며, 광역 조례화 가능성이 있음 [외부: 법제처].
  - **광주광역시**: 논의 진행 중이며, 광역 조례화 가능성이 있음 [외부: 법제처].

**3. 외부 공공데이터 활용**
- **기준일**: 2026-11-03 [외부: 법제처]
- **주요 지표**:
  - **전세 피해 관련 조례**: 이미 institutionalized된 지역(서울특별시의회, 부산광역시, 인천광역시, 대전광역시, 세종특별자치시의회 등) 외에도 울산광역시, 대구광역시, 광주광역시 등이 후속 조례 제정 가능성이 있음.
  - **전세사기 피해자 수**: 2024년 기준으로 월평균 530건, 

자유 질문 | 외부 사용: False | 출처 블록: False


## 12. FastAPI 서버 + 대시보드
`main.py`와 같은 `/api/v1/*` 엔드포인트를 위 함수로 연결하고, `frontend/` 대시보드를 함께 띄웁니다.
서버는 백그라운드 스레드에서 돌기 때문에 셀 실행이 끝나도 계속 응답합니다. 주소는 `.env`의 `HOST`·`PORT`(기본 `http://127.0.0.1:8000`).
`/api/v1/issues/{key}` 응답의 `followup_report`(9-2 셀)는 대시보드 '타 지방의회 후속 논의 흐름 및 향후 예측' 패널에 항목별로 표시됩니다.
이 패널은 `notebook_minimal/frontend`에만 추가했습니다(원본 `src/api/frontend`는 그대로).

In [23]:
import uvicorn
from fastapi import FastAPI, HTTPException, Query
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel, ConfigDict, Field, field_validator


class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


class RadarSearchRequest(StrictModel):
    query: str = Field(min_length=1, max_length=200)

    @field_validator("query")
    @classmethod
    def _trim(cls, value: str) -> str:
        if not (value := " ".join(value.split())):
            raise ValueError("query는 공백일 수 없습니다")
        return value


class ChatMessage(StrictModel):
    role: str
    content: str = Field(min_length=1, max_length=3000)

    @field_validator("role")
    @classmethod
    def _role(cls, value: str) -> str:
        if value not in {"user", "assistant"}:
            raise ValueError("role은 user 또는 assistant여야 합니다")
        return value


class ChatRequest(StrictModel):
    message: str = Field(min_length=1, max_length=4000)
    conversation_id: str | None = Field(default=None, max_length=100)
    history: list[ChatMessage] = Field(default_factory=list, max_length=8)
    top_k: int = Field(default=6, ge=1, le=20)

    @field_validator("message")
    @classmethod
    def _trim(cls, value: str) -> str:
        if not (value := " ".join(value.split())):
            raise ValueError("message는 공백일 수 없습니다")
        return value


def _iso_date(value: str | None, name: str) -> None:
    if value is not None:
        try:
            date.fromisoformat(value)
        except ValueError as error:
            raise HTTPException(422, f"{name}는 YYYY-MM-DD 형식이어야 합니다") from error


app = FastAPI(title="의정이슈레이더 API", version="1.0.0")


@app.get("/api/v1/issues")
def api_list_issues():
    return {"items": ISSUE_LIST, "count": len(ISSUE_LIST)}


@app.get("/api/v1/issues/{issue_key}")
def api_get_issue(issue_key: str):
    try:
        payload = get_issue(issue_key.casefold())
    except (ValueError, RuntimeError, OSError, psycopg.Error) as error:
        raise HTTPException(503, f"이슈 분석 실패: {type(error).__name__}: {error}") from error
    if payload is None:
        raise HTTPException(404, "대표 이슈를 찾을 수 없습니다")
    return payload


@app.post("/api/v1/radar/search")
def api_radar_search(request: RadarSearchRequest):
    try:
        return radar_search(request.query)
    except (ValueError, RuntimeError, OSError, psycopg.Error) as error:
        raise HTTPException(503, str(error)) from error


@app.get("/api/v1/documents/search")
def api_search_documents(q: str = Query(min_length=1, max_length=500), council: str | None = None,
                         year: int | None = Query(default=None, ge=2000, le=2100), committee: str | None = None,
                         date_from: str | None = None, date_to: str | None = None,
                         top_k: int = Query(default=10, ge=1, le=50)):
    _iso_date(date_from, "date_from")
    _iso_date(date_to, "date_to")
    if date_from and date_to and date_from > date_to:
        raise HTTPException(422, "date_from은 date_to보다 늦을 수 없습니다")
    filters = {"council": council, "year": year, "committee": committee, "date_from": date_from, "date_to": date_to}
    try:
        return search(q, top_k=top_k, candidate_k=max(50, top_k * 5), filters=filters)
    except (ValueError, RuntimeError, OSError, psycopg.Error) as error:
        raise HTTPException(503, str(error)) from error


@app.get("/api/v1/evidence/{chunk_id}")
def api_get_evidence(chunk_id: str):
    try:
        result = get_evidence(chunk_id)
    except Exception as error:
        raise HTTPException(503, f"원문 근거를 읽지 못했습니다: {type(error).__name__}") from error
    if result is None:
        raise HTTPException(404, "근거 청크를 찾을 수 없습니다")
    return result


@app.get("/api/v1/documents/{document_id}")
def api_get_document(document_id: str, limit: int = Query(default=100, ge=1, le=500)):
    try:
        result = get_document(document_id, limit)
    except Exception as error:
        raise HTTPException(503, f"회의록 원문을 읽지 못했습니다: {type(error).__name__}") from error
    if result is None:
        raise HTTPException(404, "회의록을 찾을 수 없습니다")
    return result


@app.get("/api/v1/chat/status")
def api_chat_status():
    result = probe_ollama(SETTINGS["ollama_url"], SETTINGS["chat_model"])
    return {"reachable": result["reachable"], "model_available": result["model_available"],
            "model": SETTINGS["chat_model"], "error": result["error"]}


@app.post("/api/v1/chat")
def api_chat(request: ChatRequest):
    try:
        return chat(request.message, [m.model_dump() for m in request.history],
                    request.top_k, request.conversation_id)
    except (ValueError, RuntimeError, OSError, psycopg.Error) as error:
        raise HTTPException(503, str(error)) from error


app.mount("/", StaticFiles(directory=BASE_DIR / "frontend", html=True), name="frontend")

HOST, PORT = SETTINGS["host"], SETTINGS["port"]
server = uvicorn.Server(uvicorn.Config(app, host=HOST, port=PORT, log_level="warning"))
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()
while not server.started and server_thread.is_alive():
    time.sleep(0.1)
print(f"대시보드: http://{HOST}:{PORT}  (API 문서: http://{HOST}:{PORT}/docs)" if server.started
      else f"서버 시작 실패: 포트 {PORT}을 다른 프로그램(이전에 띄운 노트북 서버 등)이 쓰고 있습니다. "
           ".env의 PORT를 바꾸거나 기존 서버를 종료하세요.")

대시보드: http://127.0.0.1:8010  (API 문서: http://127.0.0.1:8010/docs)


서버 응답 확인:

In [24]:
def http_get(path: str) -> Any:
    with urllib.request.urlopen(f"http://{HOST}:{PORT}{path}", timeout=30) as response:
        body = response.read().decode("utf-8")
        return json.loads(body) if "json" in response.headers.get("Content-Type", "") else body


if not server.started:
    print("이 노트북의 서버가 떠 있지 않아 확인을 건너뜁니다.")
else:
    print("GET /                  →", "대시보드 HTML" if "<html" in http_get("/").lower() else "?")
    print("GET /api/v1/issues     →", http_get("/api/v1/issues")["count"], "개 이슈")
    print("GET /api/v1/chat/status →", http_get("/api/v1/chat/status"))
    issue_payload = http_get("/api/v1/issues/jeonse_fraud")   # 8번 섹션 캐시를 써서 바로 응답
    print("GET /api/v1/issues/jeonse_fraud → 최초 관측:", issue_payload["summary"]["first_observation_councils"],
          "| 외부 후속 흐름:", issue_payload["external_followup_status"],
          "| 후속 흐름·예측 보고서 항목:", [s["title"] for s in issue_payload["followup_report"]["sections"]])
    print("  의회별 대표 발언:", len(issue_payload["council_evidence"]), "/", issue_payload["summary"]["council_count"], "개 의회")
    # 대시보드 채팅창과 같은 요청: 답변 본문 끝에 외부 출처 블록이 붙어 화면에 그대로 표시된다
    body = json.dumps({"message": "전세사기 조례를 다음에 만들 지역은?", "history": [], "top_k": 6},
                      ensure_ascii=False).encode("utf-8")
    request = urllib.request.Request(f"http://{HOST}:{PORT}/api/v1/chat", data=body, method="POST",
                                     headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=300) as response:
        chat_reply = json.loads(response.read().decode("utf-8"))
    block = chat_reply["answer"][chat_reply["answer"].find("[외부 공공데이터 출처"):]
    print("POST /api/v1/chat → 근거", chat_reply["plan"]["result_count"], "건",
          chat_reply["plan"]["evidence_basis"], "(검색창 대표 근거:", len(issue_payload["evidence"]), "건)")
    print("출처 블록:\n" + (block if "[외부 공공데이터 출처" in block else "(없음)"))

GET /                  → 대시보드 HTML
GET /api/v1/issues     → 10 개 이슈


GET /api/v1/chat/status → {'reachable': True, 'model_available': True, 'model': 'hf.co/mykor/A.X-4.0-Light-gguf:Q4_K_M', 'error': None}
GET /api/v1/issues/jeonse_fraud → 최초 관측: ['서울특별시의회'] | 외부 후속 흐름: complete | 후속 흐름·예측 보고서 항목: ['회의록에서 관측된 논의 확산 흐름', '타 지방의회 후속 논의 단계', '향후 후속 조례화·논의 진입 후보', '다음 달 논의 전망', '외부 근거를 반영한 종합 해석', '유의사항 · 출처']
  의회별 대표 발언: 17 / 17 개 의회


POST /api/v1/chat → 근거 14 건 {'search_box': 10, 'question': 6} (검색창 대표 근거: 10 건)
출처 블록:
[외부 공공데이터 출처 · live 모드 · 외부 데이터 기준일 2026-11-03 · 회의록 기준일 2025-12-31 · 참고용]
- 출처: 법제처 국가법령정보센터(www.law.go.kr) 자치법규 Open API — 관련 자치법규 45건 (공포일 기준일 이하)
- 출처: 국회사무처 열린국회정보(open.assembly.go.kr) — 관련 국회의원 발의 법률안 2건 (최근 12개월 1건)
- 출처: 통계청 KOSIS 국가통계포털, 「주택전세가격지수(시도/시/군/구)」(작성기관; 통계표 101/DT_1YL13602E), 수록시점 2026 — 지역별 정책 수요 지표: 주택 전세가격지수 (2026)
- 외부 조회 검색어: 전세 피해, 임차인 보호, 주거 피해 (회의록 원문·질문은 외부로 보내지 않음)
※ 후속 후보 점수는 미리 정한 규칙의 참고값이며 확률·인과관계가 아닙니다.


서버 종료 (대시보드를 계속 쓰려면 실행하지 마세요):

In [25]:
# server.should_exit = True
# server_thread.join(timeout=10)
# print("서버 종료:", not server_thread.is_alive())